# **🚀 Easy-RealESRGAN-Image-Upscaler**

# 📦 1. Environment Setup & Initialization

In [1]:
# ==============================================================================
# EASY-REALESRGAN RUNTIME BOOTSTRAP SYSTEM
# Environment Setup + Runtime Diagnostics + Worker Preparation
# Optimized for Kaggle / Colab + Multi GPU Architecture
# ==============================================================================

import gc
import importlib.util
import logging
import os
import platform
import shutil
import subprocess
import sys
import uuid
from datetime import datetime
from pathlib import Path


# ==============================================================================
# SESSION & RUNTIME IDENTIFIERS
# ==============================================================================

SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUNTIME_ID = str(uuid.uuid4())[:8]


# ==============================================================================
# LOGGING SYSTEM INITIALIZATION
# ==============================================================================

LOG_DIR = Path("logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "latest.log"
ERROR_LOG_FILE = LOG_DIR / "errors.log"

logger = logging.getLogger("EasyRealESRGAN")

logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter(
    "[%(asctime)s][%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

console_handler = logging.StreamHandler()
console_handler.setFormatter(formatter)

file_handler = logging.FileHandler(LOG_FILE)
file_handler.setFormatter(formatter)

error_handler = logging.FileHandler(ERROR_LOG_FILE)
error_handler.setLevel(logging.ERROR)
error_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)
logger.addHandler(error_handler)

logger.info("=" * 60)
logger.info("EASY-REALESRGAN RUNTIME INITIALIZED")
logger.info("=" * 60)

logger.info(f"Session ID : {SESSION_ID}")
logger.info(f"Runtime ID : {RUNTIME_ID}")


# ==============================================================================
# GLOBAL CONFIGURATION
# ==============================================================================

CONFIG = {

    # --------------------------------------------------------------------------
    # PROJECT
    # --------------------------------------------------------------------------

    "project_name": "Real-ESRGAN",

    "repo_url": (
        "https://github.com/xinntao/Real-ESRGAN.git"
    ),

    # --------------------------------------------------------------------------
    # RUNTIME
    # --------------------------------------------------------------------------

    "enable_memory_cleanup": True,
    "enable_runtime_optimization": True,
    "enable_worker_directories": True,
    "save_logs": True,

    # --------------------------------------------------------------------------
    # GPU
    # --------------------------------------------------------------------------

    "enable_fp16": True,
    "enable_tf32": True,
    "enable_cudnn_benchmark": True,

    # --------------------------------------------------------------------------
    # WORKERS
    # --------------------------------------------------------------------------

    "max_gpu_workers": 8,
    "worker_root_dir": "workers",

    # --------------------------------------------------------------------------
    # INSTALLATION
    # --------------------------------------------------------------------------

    "pip_quiet_install": True,
    "pip_retry_attempts": 2,
}


# ==============================================================================
# GLOBAL RUNTIME STATE
# ==============================================================================

RUNTIME_STATE = {

    "runtime_type": "unknown",

    "gpu_count": 0,

    "available_gpus": [],

    "worker_count": 0,

    "fp16_supported": False,

    "cuda_available": False,

    "torch_version": None,

    "python_version": platform.python_version(),

    "platform": platform.platform(),
}


# ==============================================================================
# RUNTIME TYPE DETECTION
# ==============================================================================

def detect_runtime_type():

    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return "kaggle"

    if "COLAB_GPU" in os.environ:
        return "colab"

    return "local"


# ==============================================================================
# GPU DETECTION SYSTEM
# ==============================================================================

def detect_available_gpus():

    logger.info("=" * 60)
    logger.info("GPU DETECTION")
    logger.info("=" * 60)

    try:

        import torch

        RUNTIME_STATE["torch_version"] = torch.__version__

        cuda_available = torch.cuda.is_available()

        RUNTIME_STATE["cuda_available"] = cuda_available

        logger.info(f"CUDA Available : {cuda_available}")

        if not cuda_available:

            logger.warning(
                "CUDA is not available. "
                "GPU acceleration disabled."
            )

            return

        gpu_count = torch.cuda.device_count()

        RUNTIME_STATE["gpu_count"] = gpu_count

        logger.info(f"Detected GPU Count : {gpu_count}")

        available_gpus = []

        for gpu_index in range(gpu_count):

            gpu_name = torch.cuda.get_device_name(gpu_index)

            gpu_properties = (
                torch.cuda.get_device_properties(gpu_index)
            )

            total_vram = round(
                gpu_properties.total_memory / (1024**3),
                2
            )

            gpu_info = {

                "index": gpu_index,

                "name": gpu_name,

                "vram_gb": total_vram,
            }

            available_gpus.append(gpu_info)

            logger.info(
                f"GPU {gpu_index} : "
                f"{gpu_name} "
                f"({total_vram} GB)"
            )

        RUNTIME_STATE["available_gpus"] = available_gpus

        worker_count = min(
            gpu_count,
            CONFIG["max_gpu_workers"]
        )

        RUNTIME_STATE["worker_count"] = worker_count

        logger.info(
            f"Initialized GPU Worker Count : "
            f"{worker_count}"
        )

        # ----------------------------------------------------------------------
        # FP16 SUPPORT CHECK
        # ----------------------------------------------------------------------

        major_capability = (
            torch.cuda.get_device_capability(0)[0]
        )

        fp16_supported = major_capability >= 7

        RUNTIME_STATE["fp16_supported"] = fp16_supported

        logger.info(
            f"FP16 Support : {fp16_supported}"
        )

    except Exception:

        logger.exception(
            "GPU detection failed"
        )


# ==============================================================================
# RUNTIME ENVIRONMENT VALIDATION
# ==============================================================================

def validate_runtime_environment():

    logger.info("=" * 60)
    logger.info("RUNTIME VALIDATION")
    logger.info("=" * 60)

    try:

        runtime_type = detect_runtime_type()

        RUNTIME_STATE["runtime_type"] = runtime_type

        logger.info(
            f"Runtime Type : {runtime_type}"
        )

        disk_usage = shutil.disk_usage("/")

        free_gb = round(
            disk_usage.free / (1024**3),
            2
        )

        logger.info(
            f"Available Disk Space : {free_gb} GB"
        )

        if free_gb < 5:

            logger.warning(
                "Low available disk space detected"
            )

    except Exception:

        logger.exception(
            "Runtime validation failed"
        )


# ==============================================================================
# WORKER DIRECTORY INITIALIZATION
# ==============================================================================

def initialize_worker_directories():

    if not CONFIG["enable_worker_directories"]:

        logger.info(
            "Worker directory initialization disabled"
        )

        return

    logger.info("=" * 60)
    logger.info("INITIALIZING WORKER DIRECTORIES")
    logger.info("=" * 60)

    try:

        worker_root = Path(
            CONFIG["worker_root_dir"]
        )

        worker_root.mkdir(
            parents=True,
            exist_ok=True
        )

        worker_count = (
            RUNTIME_STATE["worker_count"]
        )

        if worker_count == 0:

            logger.warning(
                "No GPU workers available. "
                "Worker directories skipped."
            )

            return

        for worker_id in range(worker_count):

            worker_dir = (
                worker_root / f"gpu{worker_id}"
            )

            directories = [

                worker_dir,

                worker_dir / "inputs",

                worker_dir / "outputs",

                worker_dir / "temp",

                worker_dir / "logs",
            ]

            for directory in directories:

                directory.mkdir(
                    parents=True,
                    exist_ok=True
                )

            logger.info(
                f"Worker directory initialized : "
                f"{worker_dir}"
            )

    except Exception:

        logger.exception(
            "Worker directory initialization failed"
        )


# ==============================================================================
# RUNTIME OPTIMIZATION
# ==============================================================================

def optimize_runtime_environment():

    if not CONFIG["enable_runtime_optimization"]:

        logger.info(
            "Runtime optimization disabled"
        )

        return

    logger.info("=" * 60)
    logger.info("RUNTIME OPTIMIZATION")
    logger.info("=" * 60)

    try:

        import torch

        if torch.cuda.is_available():

            # ------------------------------------------------------------------
            # CUDNN BENCHMARK
            # ------------------------------------------------------------------

            torch.backends.cudnn.benchmark = (
                CONFIG["enable_cudnn_benchmark"]
            )

            logger.info(
                f"CUDNN Benchmark : "
                f"{torch.backends.cudnn.benchmark}"
            )

            # ------------------------------------------------------------------
            # TF32 SUPPORT
            # ------------------------------------------------------------------

            if CONFIG["enable_tf32"]:

                torch.backends.cuda.matmul.allow_tf32 = True

                torch.backends.cudnn.allow_tf32 = True

                logger.info(
                    "TF32 optimization enabled"
                )

            # ------------------------------------------------------------------
            # MEMORY CLEANUP
            # ------------------------------------------------------------------

            gc.collect()

            torch.cuda.empty_cache()

            logger.info(
                "Initial CUDA memory cleanup completed"
            )

    except Exception:

        logger.exception(
            "Runtime optimization failed"
        )


# ==============================================================================
# DEPENDENCY INSTALLATION SYSTEM
# ==============================================================================

def install_dependencies():

    logger.info("=" * 60)
    logger.info("DEPENDENCY INSTALLATION")
    logger.info("=" * 60)

    pip_cmd = [sys.executable, "-m", "pip", "install"]

    if CONFIG["pip_quiet_install"]:
        pip_cmd.append("-q")

    install_commands = [

        ["-r", "requirements.txt"],

        ["gfpgan", "facexlib", "gdown"],

        ["-e", "."]
    ]

    for command in install_commands:

        success = False

        for attempt in range(
            CONFIG["pip_retry_attempts"]
        ):

            try:

                logger.info(
                    f"Installing dependency : "
                    f"{' '.join(command)} "
                    f"(Attempt {attempt + 1})"
                )

                subprocess.run(
                    pip_cmd + command,
                    check=True
                )

                success = True

                break

            except subprocess.CalledProcessError:

                logger.exception(
                    f"Dependency installation failed : "
                    f"{' '.join(command)}"
                )

        if not success:

            raise RuntimeError(
                f"Failed dependency installation : "
                f"{' '.join(command)}"
            )


# ==============================================================================
# PYTHON COMPATIBILITY PATCH
# ==============================================================================

def apply_python_compatibility_patch():

    logger.info("=" * 60)
    logger.info("PYTHON COMPATIBILITY PATCH")
    logger.info("=" * 60)

    try:

        spec = importlib.util.find_spec("basicsr")

        if not (spec and spec.origin):

            logger.warning(
                "Library 'basicsr' not found. "
                "Compatibility patch skipped."
            )

            return

        basicsr_path = Path(spec.origin).parent

        target_file = (
            basicsr_path /
            "data" /
            "degradations.py"
        )

        if not target_file.exists():

            logger.error(
                f"Patch target file not found : "
                f"{target_file}"
            )

            return

        content = target_file.read_text(
            encoding="utf-8"
        )

        old_import = (
            "from torchvision.transforms.functional_tensor "
            "import rgb_to_grayscale"
        )

        new_import = (
            "from torchvision.transforms.functional "
            "import rgb_to_grayscale"
        )

        if old_import in content:

            content = content.replace(
                old_import,
                new_import
            )

            target_file.write_text(
                content,
                encoding="utf-8"
            )

            logger.info(
                "Compatibility patch applied successfully"
            )

        else:

            logger.info(
                "Compatibility patch already applied "
                "or not required"
            )

    except Exception:

        logger.exception(
            "Compatibility patch failed"
        )


# ==============================================================================
# PROJECT REPOSITORY SETUP
# ==============================================================================

def setup_repository():

    logger.info("=" * 60)
    logger.info("REPOSITORY SETUP")
    logger.info("=" * 60)

    project_name = CONFIG["project_name"]

    project_dir = Path(project_name)

    current_dir = Path.cwd()

    try:

        if current_dir.name == project_name:

            logger.info(
                f"Already inside repository : "
                f"{project_name}"
            )

        elif project_dir.exists():

            logger.info(
                f"Repository found : "
                f"{project_name}"
            )

            os.chdir(project_dir)

        else:

            logger.info(
                f"Cloning repository : "
                f"{project_name}"
            )

            subprocess.run(

                [
                    "git",
                    "clone",
                    CONFIG["repo_url"],
                    "-q"
                ],

                check=True
            )

            os.chdir(project_dir)

        current_path = str(Path.cwd())

        if current_path not in sys.path:

            sys.path.append(current_path)

        logger.info(
            f"Working Directory : "
            f"{Path.cwd()}"
        )

    except Exception:

        logger.exception(
            "Repository setup failed"
        )

        raise


# ==============================================================================
# FINAL RUNTIME SUMMARY
# ==============================================================================

def print_runtime_summary():

    logger.info("=" * 60)
    logger.info("RUNTIME SUMMARY")
    logger.info("=" * 60)

    for key, value in RUNTIME_STATE.items():

        logger.info(f"{key} : {value}")


# ==============================================================================
# MAIN ENVIRONMENT INITIALIZATION
# ==============================================================================

def initialize_environment():

    logger.info("=" * 60)
    logger.info(
        "STARTING EASY-REALESRGAN ENVIRONMENT SETUP"
    )
    logger.info("=" * 60)

    try:

        # ----------------------------------------------------------------------
        # VALIDATE RUNTIME
        # ----------------------------------------------------------------------

        validate_runtime_environment()

        # ----------------------------------------------------------------------
        # DETECT GPUS
        # ----------------------------------------------------------------------

        detect_available_gpus()

        # ----------------------------------------------------------------------
        # OPTIMIZE RUNTIME
        # ----------------------------------------------------------------------

        optimize_runtime_environment()

        # ----------------------------------------------------------------------
        # REPOSITORY SETUP
        # ----------------------------------------------------------------------

        setup_repository()

        # ----------------------------------------------------------------------
        # INSTALL DEPENDENCIES
        # ----------------------------------------------------------------------

        install_dependencies()

        # ----------------------------------------------------------------------
        # WORKER DIRECTORIES
        # ----------------------------------------------------------------------

        initialize_worker_directories()

        # ----------------------------------------------------------------------
        # APPLY COMPATIBILITY PATCH
        # ----------------------------------------------------------------------

        apply_python_compatibility_patch()

        # ----------------------------------------------------------------------
        # FINAL MEMORY CLEANUP
        # ----------------------------------------------------------------------

        if CONFIG["enable_memory_cleanup"]:

            try:

                import torch

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                logger.info(
                    "Final memory cleanup completed"
                )

            except Exception:

                logger.exception(
                    "Final memory cleanup failed"
                )

        # ----------------------------------------------------------------------
        # FINAL SUMMARY
        # ----------------------------------------------------------------------

        print_runtime_summary()

        logger.info("=" * 60)
        logger.info(
            "EASY-REALESRGAN ENVIRONMENT READY"
        )
        logger.info("=" * 60)

    except Exception:

        logger.exception(
            "Fatal environment initialization failure"
        )


# ==============================================================================
# EXECUTION
# ==============================================================================

if __name__ == "__main__":

    initialize_environment()

[2026-05-18 12:11:29][INFO] ============================================================
[2026-05-18 12:11:29][INFO] EASY-REALESRGAN RUNTIME INITIALIZED
[2026-05-18 12:11:29][INFO] ============================================================
[2026-05-18 12:11:29][INFO] Session ID : 20260518_121129
[2026-05-18 12:11:29][INFO] Runtime ID : 973ff78c
[2026-05-18 12:11:29][INFO] ============================================================
[2026-05-18 12:11:29][INFO] STARTING EASY-REALESRGAN ENVIRONMENT SETUP
[2026-05-18 12:11:29][INFO] ============================================================
[2026-05-18 12:11:29][INFO] ============================================================
[2026-05-18 12:11:29][INFO] RUNTIME VALIDATION
[2026-05-18 12:11:29][INFO] ============================================================
[2026-05-18 12:11:29][INFO] Runtime Type : kaggle
[2026-05-18 12:11:29][INFO] Available Disk Space : 1220.51 GB
[2026-05-18 12:11:29][INFO] =====================================

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 13.4 MB/s eta 0:00:00


[2026-05-18 12:11:52][INFO] Installing dependency : gfpgan facexlib gdown (Attempt 1)
[2026-05-18 12:11:55][INFO] Installing dependency : -e . (Attempt 1)
[2026-05-18 12:12:02][INFO] ============================================================
[2026-05-18 12:12:02][INFO] INITIALIZING WORKER DIRECTORIES
[2026-05-18 12:12:02][INFO] ============================================================
[2026-05-18 12:12:02][INFO] Worker directory initialized : workers/gpu0
[2026-05-18 12:12:02][INFO] Worker directory initialized : workers/gpu1
[2026-05-18 12:12:02][INFO] ============================================================
[2026-05-18 12:12:02][INFO] PYTHON COMPATIBILITY PATCH
[2026-05-18 12:12:02][INFO] ============================================================
[2026-05-18 12:12:02][INFO] Compatibility patch applied successfully
[2026-05-18 12:12:02][INFO] Final memory cleanup completed
[2026-05-18 12:12:02][INFO] ============================================================
[2026-05-18 1

# 📥 2. Smart Model Downloader

In [2]:
# ==============================================================================
# ADVANCED MODEL REGISTRY & DOWNLOADER SYSTEM
# Download + Validation + Retry + Integrity + Multi Worker Preparation
# Optimized for Kaggle / Colab + Multi GPU Architecture
# ==============================================================================

import hashlib
import shutil
import time
from pathlib import Path

import requests
from tqdm.auto import tqdm


# ==============================================================================
# MODEL REGISTRY CONFIGURATION
# ==============================================================================

WEIGHTS_DIR = Path("weights")

WEIGHTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_REGISTRY = {

    # --------------------------------------------------------------------------
    # GENERAL PHOTO MODEL
    # --------------------------------------------------------------------------

    "RealESRGAN_x4plus.pth": {

        "url": (
            "https://github.com/xinntao/Real-ESRGAN/"
            "releases/download/v0.1.0/"
            "RealESRGAN_x4plus.pth"
        ),

        "category": "general",

        "scale": 4,

        "recommended_tile": 200,

        "fp16_recommended": True,
    },

    # --------------------------------------------------------------------------
    # ANIME MODEL
    # --------------------------------------------------------------------------

    "realesr-animevideov3.pth": {

        "url": (
            "https://github.com/xinntao/Real-ESRGAN/"
            "releases/download/v0.2.5.0/"
            "realesr-animevideov3.pth"
        ),

        "category": "anime",

        "scale": 4,

        "recommended_tile": 400,

        "fp16_recommended": True,
    },

    # --------------------------------------------------------------------------
    # X2 MODEL
    # --------------------------------------------------------------------------

    "RealESRGAN_x2plus.pth": {

        "url": (
            "https://github.com/xinntao/Real-ESRGAN/"
            "releases/download/v0.2.1/"
            "RealESRGAN_x2plus.pth"
        ),

        "category": "general",

        "scale": 2,

        "recommended_tile": 600,

        "fp16_recommended": True,
    },
}


# ==============================================================================
# DOWNLOAD CONFIGURATION
# ==============================================================================

DOWNLOAD_CONFIG = {

    "chunk_size": 8192,

    "request_timeout": 60,

    "head_timeout": 15,

    "max_retries": 3,

    "retry_delay": 3,

    "enable_partial_cleanup": True,

    "enable_worker_model_links": True,
}


# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def format_size(size_bytes: int) -> str:

    return (
        f"{size_bytes / (1024 ** 2):.2f} MB"
    )


def calculate_file_hash(
    file_path: Path,
    algorithm: str = "md5"
) -> str:
    """
    Generate file hash for integrity validation.
    """

    hash_function = hashlib.new(algorithm)

    with file_path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(8192),
            b""
        ):

            hash_function.update(chunk)

    return hash_function.hexdigest()


# ==============================================================================
# REMOTE FILE VALIDATION
# ==============================================================================

def get_remote_file_metadata(url: str) -> dict:
    """
    Retrieve remote metadata using HEAD request.
    """

    try:

        response = requests.head(
            url,
            allow_redirects=True,
            timeout=DOWNLOAD_CONFIG["head_timeout"]
        )

        response.raise_for_status()

        content_length = int(
            response.headers.get(
                "content-length",
                0
            )
        )

        return {

            "size": content_length,

            "content_type": response.headers.get(
                "content-type",
                "unknown"
            ),
        }

    except Exception:

        logger.exception(
            f"Failed to retrieve metadata: {url}"
        )

        return {

            "size": 0,

            "content_type": "unknown",
        }


# ==============================================================================
# LOCAL FILE VALIDATION
# ==============================================================================

def validate_existing_model(
    model_path: Path,
    remote_size: int
) -> bool:
    """
    Validate local model using:
    - existence
    - file size
    - corruption check
    """

    if not model_path.exists():

        return False

    try:

        local_size = model_path.stat().st_size

        if local_size <= 0:

            logger.warning(
                f"Corrupted empty model detected: "
                f"{model_path.name}"
            )

            return False

        if remote_size > 0:

            if local_size != remote_size:

                logger.warning(
                    f"Model validation mismatch: "
                    f"{model_path.name} "
                    f"(Local={local_size} | "
                    f"Remote={remote_size})"
                )

                return False

        logger.info(
            f"Valid model detected: "
            f"{model_path.name} "
            f"({format_size(local_size)})"
        )

        return True

    except Exception:

        logger.exception(
            f"Model validation failed: "
            f"{model_path.name}"
        )

        return False


# ==============================================================================
# DOWNLOAD ENGINE
# ==============================================================================

def download_model(
    model_name: str,
    model_info: dict
) -> bool:
    """
    Advanced model downloader with:
    - retry handling
    - validation
    - integrity checks
    - speed tracking
    - partial cleanup
    """

    model_url = model_info["url"]

    model_path = WEIGHTS_DIR / model_name

    logger.info("=" * 60)
    logger.info(
        f"PREPARING MODEL: {model_name}"
    )
    logger.info("=" * 60)

    remote_metadata = (
        get_remote_file_metadata(model_url)
    )

    remote_size = remote_metadata["size"]

    logger.info(
        f"Model Category : "
        f"{model_info['category']}"
    )

    logger.info(
        f"Recommended Tile : "
        f"{model_info['recommended_tile']}"
    )

    logger.info(
        f"FP16 Recommended : "
        f"{model_info['fp16_recommended']}"
    )

    if validate_existing_model(
        model_path,
        remote_size
    ):

        logger.info(
            f"Skipping download: {model_name}"
        )

        return True

    # --------------------------------------------------------------------------
    # REMOVE INVALID FILE
    # --------------------------------------------------------------------------

    if model_path.exists():

        try:

            model_path.unlink()

            logger.warning(
                f"Removed invalid model file: "
                f"{model_name}"
            )

        except Exception:

            logger.exception(
                f"Failed to remove invalid file: "
                f"{model_name}"
            )

    # --------------------------------------------------------------------------
    # DOWNLOAD RETRY LOOP
    # --------------------------------------------------------------------------

    for attempt in range(
        1,
        DOWNLOAD_CONFIG["max_retries"] + 1
    ):

        try:

            logger.info(
                f"Download attempt "
                f"{attempt}/"
                f"{DOWNLOAD_CONFIG['max_retries']}"
            )

            response = requests.get(
                model_url,
                stream=True,
                timeout=DOWNLOAD_CONFIG[
                    "request_timeout"
                ]
            )

            response.raise_for_status()

            total_size = int(
                response.headers.get(
                    "content-length",
                    0
                )
            )

            start_time = time.time()

            with model_path.open("wb") as file, tqdm(

                desc=model_name,

                total=total_size,

                unit="iB",

                unit_scale=True,

                unit_divisor=1024,

            ) as progress_bar:

                for chunk in response.iter_content(

                    chunk_size=DOWNLOAD_CONFIG[
                        "chunk_size"
                    ]

                ):

                    if chunk:

                        written = file.write(chunk)

                        progress_bar.update(written)

            elapsed_time = (
                time.time() - start_time
            )

            downloaded_size = (
                model_path.stat().st_size
            )

            # ------------------------------------------------------------------
            # VALIDATION
            # ------------------------------------------------------------------

            if remote_size > 0:

                if downloaded_size != remote_size:

                    logger.error(
                        f"Downloaded model validation failed: "
                        f"{model_name}"
                    )

                    continue

            # ------------------------------------------------------------------
            # HASH GENERATION
            # ------------------------------------------------------------------

            model_hash = calculate_file_hash(
                model_path
            )

            logger.info(
                f"Download completed successfully"
            )

            logger.info(
                f"Model Size : "
                f"{format_size(downloaded_size)}"
            )

            logger.info(
                f"Elapsed Time : "
                f"{elapsed_time:.2f}s"
            )

            logger.info(
                f"Model Hash : "
                f"{model_hash}"
            )

            return True

        except requests.exceptions.RequestException:

            logger.exception(
                f"Network download failure: "
                f"{model_name}"
            )

        except Exception:

            logger.exception(
                f"Unexpected download failure: "
                f"{model_name}"
            )

        # ----------------------------------------------------------------------
        # CLEANUP FAILED PARTIAL FILE
        # ----------------------------------------------------------------------

        if (
            DOWNLOAD_CONFIG[
                "enable_partial_cleanup"
            ]
            and
            model_path.exists()
        ):

            try:

                model_path.unlink()

                logger.warning(
                    f"Removed partial download: "
                    f"{model_name}"
                )

            except Exception:

                logger.exception(
                    f"Failed partial cleanup: "
                    f"{model_name}"
                )

        # ----------------------------------------------------------------------
        # RETRY DELAY
        # ----------------------------------------------------------------------

        if (
            attempt <
            DOWNLOAD_CONFIG["max_retries"]
        ):

            retry_delay = (
                DOWNLOAD_CONFIG[
                    "retry_delay"
                ]
            )

            logger.warning(
                f"Retrying download in "
                f"{retry_delay} seconds..."
            )

            time.sleep(retry_delay)

    logger.error(
        f"Failed to download model: "
        f"{model_name}"
    )

    return False


# ==============================================================================
# WORKER MODEL LINK INITIALIZATION
# ==============================================================================

def initialize_worker_model_links():
    """
    Prepare model access for multi GPU workers.
    """

    if not DOWNLOAD_CONFIG[
        "enable_worker_model_links"
    ]:

        return

    logger.info("=" * 60)
    logger.info(
        "INITIALIZING WORKER MODEL ACCESS"
    )
    logger.info("=" * 60)

    try:

        worker_root = Path(
            CONFIG["worker_root_dir"]
        )

        if not worker_root.exists():

            logger.warning(
                "Worker directory not found"
            )

            return

        worker_count = (
            RUNTIME_STATE["worker_count"]
        )

        for worker_id in range(worker_count):

            worker_dir = (
                worker_root /
                f"gpu{worker_id}"
            )

            worker_weights_dir = (
                worker_dir / "weights"
            )

            worker_weights_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            logger.info(
                f"Worker model access ready: "
                f"gpu{worker_id}"
            )

    except Exception:

        logger.exception(
            "Worker model initialization failed"
        )


# ==============================================================================
# MODEL SUMMARY REPORT
# ==============================================================================

def print_model_summary():

    logger.info("=" * 60)
    logger.info("MODEL REGISTRY SUMMARY")
    logger.info("=" * 60)

    for model_name, model_info in MODEL_REGISTRY.items():

        logger.info(
            f"{model_name} | "
            f"Scale={model_info['scale']}x | "
            f"Category={model_info['category']}"
        )


# ==============================================================================
# MAIN MODEL SETUP
# ==============================================================================

def setup_models():

    logger.info("=" * 60)
    logger.info(
        "STARTING ADVANCED MODEL SETUP"
    )
    logger.info("=" * 60)

    successful_models = []

    failed_models = []

    try:

        print_model_summary()

        # ----------------------------------------------------------------------
        # DOWNLOAD MODELS
        # ----------------------------------------------------------------------

        for model_name, model_info in (

            MODEL_REGISTRY.items()

        ):

            success = download_model(
                model_name=model_name,
                model_info=model_info
            )

            if success:

                successful_models.append(
                    model_name
                )

            else:

                failed_models.append(
                    model_name
                )

        # ----------------------------------------------------------------------
        # INITIALIZE WORKER MODEL ACCESS
        # ----------------------------------------------------------------------

        initialize_worker_model_links()

        # ----------------------------------------------------------------------
        # FINAL SUMMARY
        # ----------------------------------------------------------------------

        logger.info("=" * 60)
        logger.info("MODEL SETUP SUMMARY")
        logger.info("=" * 60)

        logger.info(
            f"Successful Models : "
            f"{len(successful_models)}"
        )

        logger.info(
            f"Failed Models : "
            f"{len(failed_models)}"
        )

        if successful_models:

            logger.info(
                f"Downloaded Models : "
                f"{', '.join(successful_models)}"
            )

        if failed_models:

            logger.warning(
                f"Failed Models : "
                f"{', '.join(failed_models)}"
            )

        logger.info(
            "Advanced model setup completed"
        )

    except Exception:

        logger.exception(
            "Fatal model setup failure"
        )


# ==============================================================================
# EXECUTION
# ==============================================================================

if __name__ == "__main__":

    setup_models()

[2026-05-18 12:12:03][INFO] ============================================================
[2026-05-18 12:12:03][INFO] STARTING ADVANCED MODEL SETUP
[2026-05-18 12:12:03][INFO] ============================================================
[2026-05-18 12:12:03][INFO] ============================================================
[2026-05-18 12:12:03][INFO] MODEL REGISTRY SUMMARY
[2026-05-18 12:12:03][INFO] ============================================================
[2026-05-18 12:12:03][INFO] RealESRGAN_x4plus.pth | Scale=4x | Category=general
[2026-05-18 12:12:03][INFO] realesr-animevideov3.pth | Scale=4x | Category=anime
[2026-05-18 12:12:03][INFO] RealESRGAN_x2plus.pth | Scale=2x | Category=general
[2026-05-18 12:12:03][INFO] ============================================================
[2026-05-18 12:12:03][INFO] PREPARING MODEL: RealESRGAN_x4plus.pth
[2026-05-18 12:12:03][INFO] ============================================================
[2026-05-18 12:12:03][INFO] Model Category : gene

RealESRGAN_x4plus.pth:   0%|          | 0.00/63.9M [00:00<?, ?iB/s]

[2026-05-18 12:12:04][INFO] Download completed successfully
[2026-05-18 12:12:04][INFO] Model Size : 63.94 MB
[2026-05-18 12:12:04][INFO] Elapsed Time : 0.31s
[2026-05-18 12:12:04][INFO] Model Hash : 99ec365d4afad750833258a1a24f44ca
[2026-05-18 12:12:04][INFO] ============================================================
[2026-05-18 12:12:04][INFO] PREPARING MODEL: realesr-animevideov3.pth
[2026-05-18 12:12:04][INFO] ============================================================
[2026-05-18 12:12:04][INFO] Model Category : anime
[2026-05-18 12:12:04][INFO] Recommended Tile : 400
[2026-05-18 12:12:04][INFO] FP16 Recommended : True
[2026-05-18 12:12:04][INFO] Download attempt 1/3


realesr-animevideov3.pth:   0%|          | 0.00/2.39M [00:00<?, ?iB/s]

[2026-05-18 12:12:05][INFO] Download completed successfully
[2026-05-18 12:12:05][INFO] Model Size : 2.39 MB
[2026-05-18 12:12:05][INFO] Elapsed Time : 0.20s
[2026-05-18 12:12:05][INFO] Model Hash : 022f0b7f79ceb1ae1ad245d0296a27c7
[2026-05-18 12:12:05][INFO] ============================================================
[2026-05-18 12:12:05][INFO] PREPARING MODEL: RealESRGAN_x2plus.pth
[2026-05-18 12:12:05][INFO] ============================================================
[2026-05-18 12:12:05][INFO] Model Category : general
[2026-05-18 12:12:05][INFO] Recommended Tile : 600
[2026-05-18 12:12:05][INFO] FP16 Recommended : True
[2026-05-18 12:12:05][INFO] Download attempt 1/3


RealESRGAN_x2plus.pth:   0%|          | 0.00/64.0M [00:00<?, ?iB/s]

[2026-05-18 12:12:06][INFO] Download completed successfully
[2026-05-18 12:12:06][INFO] Model Size : 63.96 MB
[2026-05-18 12:12:06][INFO] Elapsed Time : 0.33s
[2026-05-18 12:12:06][INFO] Model Hash : f2a8176bb1c58ede774b27bdfebbf2c7
[2026-05-18 12:12:06][INFO] ============================================================
[2026-05-18 12:12:06][INFO] INITIALIZING WORKER MODEL ACCESS
[2026-05-18 12:12:06][INFO] ============================================================
[2026-05-18 12:12:06][INFO] Worker model access ready: gpu0
[2026-05-18 12:12:06][INFO] Worker model access ready: gpu1
[2026-05-18 12:12:06][INFO] ============================================================
[2026-05-18 12:12:06][INFO] MODEL SETUP SUMMARY
[2026-05-18 12:12:06][INFO] ============================================================
[2026-05-18 12:12:06][INFO] Successful Models : 3
[2026-05-18 12:12:06][INFO] Failed Models : 0
[2026-05-18 12:12:06][INFO] Downloaded Models : RealESRGAN_x4plus.pth, realesr-animevi

# 🧠 3. Core Engine & Configuration

In [3]:
# ==============================================================================
# ADVANCED CORE ENGINE & CONFIGURATION SYSTEM
# Real-ESRGAN Inference Engine + Adaptive Processing + Multi GPU Ready
# Optimized for Kaggle / Colab + Open Source Scalability
# ==============================================================================

import gc
import shutil
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple

import cv2
import torch


logger.info("=" * 60)
logger.info("INITIALIZING ADVANCED CORE ENGINE")
logger.info("=" * 60)


# ==============================================================================
# QUALITY CONFIGURATION SYSTEM
# ==============================================================================

class QualityConfig:
    """
    Centralized quality configuration system.
    Optimized for:
    - Kaggle
    - Colab
    - Multi GPU
    - Batch Processing
    """

    # --------------------------------------------------------------------------
    # MODEL REGISTRY
    # --------------------------------------------------------------------------

    MODELS = {

        "general": {

            "x4": (
                "RealESRGAN_x4plus.pth",
                "RealESRGAN_x4plus"
            ),

            "x2": (
                "RealESRGAN_x2plus.pth",
                "RealESRGAN_x2plus"
            ),
        },

        "anime": {

            "x4": (
                "realesr-animevideov3.pth",
                "realesr-animevideov3"
            ),

            "x4_alt": (
                "RealESRGAN_x4plus_anime_6B.pth",
                "RealESRGAN_x4plus_anime_6B"
            ),

            "x2": (
                "RealESRGAN_x2plus.pth",
                "RealESRGAN_x2plus"
            ),
        },
    }

    # --------------------------------------------------------------------------
    # QUALITY PRESETS
    # --------------------------------------------------------------------------

    PRESETS = {

        # ----------------------------------------------------------------------
        # HIGHEST QUALITY
        # ----------------------------------------------------------------------

        "ultra": {

            "fp32": True,

            "tile": 200,

            "face_enhance": True,

            "timeout": 600,

            "description": (
                "Highest quality preset "
                "with safer VRAM handling"
            ),
        },

        # ----------------------------------------------------------------------
        # HIGH QUALITY
        # ----------------------------------------------------------------------

        "high": {

            "fp32": False,

            "tile": 400,

            "face_enhance": False,

            "timeout": 480,

            "description": (
                "High quality balanced preset"
            ),
        },

        # ----------------------------------------------------------------------
        # DEFAULT RECOMMENDED
        # ----------------------------------------------------------------------

        "balanced": {

            "fp32": False,

            "tile": 600,

            "face_enhance": False,

            "timeout": 360,

            "description": (
                "Recommended preset "
                "for Kaggle & T4"
            ),
        },

        # ----------------------------------------------------------------------
        # FASTEST
        # ----------------------------------------------------------------------

        "fast": {

            "fp32": False,

            "tile": 800,

            "face_enhance": False,

            "timeout": 240,

            "description": (
                "Fastest low VRAM preset"
            ),
        },
    }

    # --------------------------------------------------------------------------
    # ADAPTIVE TILE FALLBACK
    # --------------------------------------------------------------------------

    TILE_CANDIDATES = [
        800,
        600,
        400,
        200,
        100,
    ]


# ==============================================================================
# ENGINE CONFIGURATION
# ==============================================================================

ENGINE_CONFIG = {

    "enable_output_validation": True,

    "enable_image_read_validation": True,

    "enable_memory_cleanup": True,

    "enable_stdout_logging": True,

    "enable_stderr_logging": True,

    "enable_adaptive_retry": True,

    "adaptive_retry_delay": 2,

    "max_retry_attempts": 5,
}


# ==============================================================================
# ENGINE RUNTIME METRICS
# ==============================================================================

@dataclass
class InferenceMetrics:

    success: bool

    elapsed_time: float

    tile_size: int

    return_code: int

    output_exists: bool

    stdout: str

    stderr: str


# ==============================================================================
# REFACTORED CORE UPSCALER ENGINE (STATELESS FOR MULTI-GPU)
# ==============================================================================

class RealESRGANUpscaler:

    def __init__(self):
        self.weights_dir = Path("weights")
        logger.info("Core engine initialized (Stateless for parallel execution)")

    # ==========================================================================
    # MODEL RESOLUTION
    # ==========================================================================
    def resolve_model(self, image_type: str, scale: int) -> Tuple[str, str]:
        model_key = "x4" if scale == 4 else "x2"
        
        if image_type == "anime":
            target_model = QualityConfig.MODELS["anime"][model_key][0]
            if not (self.weights_dir / target_model).exists():
                logger.warning("Primary anime model missing. Using fallback anime model.")
                model_key = "x4_alt"
            return QualityConfig.MODELS["anime"][model_key]
            
        return QualityConfig.MODELS["general"][model_key]

    # ==========================================================================
    # OUTPUT VALIDATION
    # ==========================================================================
    def validate_output_file(self, output_path: Path) -> bool:
        try:
            if not output_path.exists() or output_path.stat().st_size <= 0:
                return False
                
            if ENGINE_CONFIG.get("enable_image_read_validation", True):
                image = cv2.imread(str(output_path))
                if image is None:
                    return False
            return True
        except Exception:
            logger.exception(f"Output validation failed for {output_path.name}")
            return False

    # ==========================================================================
    # MEMORY CLEANUP
    # ==========================================================================
    def cleanup_memory(self):
        if not ENGINE_CONFIG.get("enable_memory_cleanup", True):
            return
        try:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    # ==========================================================================
    # INFERENCE EXECUTION
    # ==========================================================================
    def execute_inference(
        self,
        input_path: Path,
        output_dir: Path,
        model_name: str,
        scale: int,
        tile_size: int,
        preset: dict,
        timeout: int,
        gpu_id: int = 0
    ) -> InferenceMetrics:
        
        cmd = [
            sys.executable,
            "inference_realesrgan.py",
            "-i", str(input_path),
            "-o", str(output_dir),
            "-n", model_name,
            "-s", str(scale),
            "-t", str(tile_size),
        ]

        if preset["fp32"]:
            cmd.append("--fp32")
        if preset["face_enhance"]:
            cmd.append("--face_enhance")

        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)

        try:
            start_time = time.time()
            result = subprocess.run(
                cmd, capture_output=True, text=True, timeout=timeout, env=env
            )
            elapsed_time = time.time() - start_time
            
            output_files = list(output_dir.iterdir())
            output_exists = len(output_files) > 0
            success = (result.returncode == 0 and output_exists)

            return InferenceMetrics(
                success=success,
                elapsed_time=elapsed_time,
                tile_size=tile_size,
                return_code=result.returncode,
                output_exists=output_exists,
                stdout=result.stdout.strip(),
                stderr=result.stderr.strip(),
            )
        except subprocess.TimeoutExpired:
            return InferenceMetrics(False, timeout, tile_size, -1, False, "", "Timeout exceeded")
        except Exception as e:
            return InferenceMetrics(False, 0, tile_size, -1, False, "", str(e))

    # ==========================================================================
    # MAIN UPSCALE PROCESS
    # ==========================================================================
    def run(
        self,
        image_path: Path,
        isolated_output_dir: Path,
        image_type: str = "general",
        scale: int = 4,
        quality: str = "balanced",
        output_name: Optional[str] = None,
        gpu_id: int = 0
    ) -> Optional[Tuple[Path, str]]:
        
        preset = QualityConfig.PRESETS.get(quality, QualityConfig.PRESETS["balanced"])
        model_file, model_name = self.resolve_model(image_type=image_type, scale=scale)

        tile_candidates = [preset["tile"]] + [
            t for t in QualityConfig.TILE_CANDIDATES if t != preset["tile"]
        ]

        final_metrics = None

        for attempt_index, tile_size in enumerate(tile_candidates, start=1):
            metrics = self.execute_inference(
                input_path=image_path,
                output_dir=isolated_output_dir,
                model_name=model_name,
                scale=scale,
                tile_size=tile_size,
                preset=preset,
                timeout=preset["timeout"],
                gpu_id=gpu_id
            )
            final_metrics = metrics

            if metrics.success:
                break
                
            self.cleanup_memory()
            
            if ENGINE_CONFIG.get("enable_adaptive_retry", True):
                time.sleep(ENGINE_CONFIG.get("adaptive_retry_delay", 2))

        if not final_metrics or not final_metrics.success:
            return None

        output_files = list(isolated_output_dir.iterdir())
        if not output_files:
            return None

        temp_result = output_files[0]
        if not self.validate_output_file(temp_result):
            return None

        final_stem = output_name if output_name else temp_result.stem
        final_name = f"{final_stem}{temp_result.suffix}"
        
        return temp_result, final_name


# ==============================================================================
# GLOBAL ENGINE INSTANCE
# ==============================================================================

# Global instance is now completely thread-safe because it holds no I/O state
upscaler_engine = RealESRGANUpscaler()

# ==============================================================================
# MAIN UPSCALE FUNCTION
# ==============================================================================

def upscale_image(
    image_path: Path,
    output_folder: Path,
    isolated_output_dir: Path,
    **kwargs
) -> Optional[Path]:

    output_folder.mkdir(parents=True, exist_ok=True)
    isolated_output_dir.mkdir(parents=True, exist_ok=True)

    try:
        result = upscaler_engine.run(
            image_path=image_path,
            isolated_output_dir=isolated_output_dir,
            **kwargs
        )

        if not result:
            return None

        temp_path, final_name = result
        final_path = output_folder / final_name

        counter = 1
        while final_path.exists():
            final_path = output_folder / f"{Path(final_name).stem}_{counter}{Path(final_name).suffix}"
            counter += 1

        shutil.move(str(temp_path), str(final_path))
        return final_path
    
    except Exception:
        logger.exception(f"Fatal upscale exception: {image_path.name}")
        return None
    finally:
        upscaler_engine.cleanup_memory()


logger.info("=" * 60)
logger.info(
    "ADVANCED CORE ENGINE INITIALIZED"
)
logger.info("=" * 60)

[2026-05-18 12:12:06][INFO] ============================================================
[2026-05-18 12:12:06][INFO] INITIALIZING ADVANCED CORE ENGINE
[2026-05-18 12:12:06][INFO] ============================================================
[2026-05-18 12:12:06][INFO] Core engine initialized (Stateless for parallel execution)
[2026-05-18 12:12:06][INFO] ============================================================
[2026-05-18 12:12:06][INFO] ADVANCED CORE ENGINE INITIALIZED
[2026-05-18 12:12:06][INFO] ============================================================


# ⚙️ 4. GPU Worker Manager

In [4]:
# ==============================================================================
# GPU WORKER MANAGER SYSTEM
# Multi GPU Worker Allocation + Worker Isolation + Queue Preparation
# Optimized for Kaggle / Colab + T4 x2 Architecture
# ==============================================================================

import os
import queue
import threading
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional


logger.info("=" * 60)
logger.info("INITIALIZING GPU WORKER MANAGER")
logger.info("=" * 60)


# ==============================================================================
# GPU WORKER CONFIGURATION
# ==============================================================================

GPU_WORKER_CONFIG = {

    # --------------------------------------------------------------------------
    # WORKER CONTROL
    # --------------------------------------------------------------------------

    "enable_multi_gpu": True,

    "enable_worker_isolation": True,

    "enable_worker_logging": True,

    "enable_worker_health_check": True,

    # --------------------------------------------------------------------------
    # WORKER DIRECTORIES
    # --------------------------------------------------------------------------

    "worker_root_dir": "workers",

    "shared_output_dir": "batch_results",

    # --------------------------------------------------------------------------
    # QUEUE
    # --------------------------------------------------------------------------

    "max_queue_size": 10000,

    "worker_timeout": 600,
}


# ==============================================================================
# WORKER DATA STRUCTURE
# ==============================================================================

@dataclass
class GPUWorker:

    worker_id: int

    gpu_id: int

    gpu_name: str

    vram_gb: float

    worker_uuid: str = field(

        default_factory=lambda:
        str(uuid.uuid4())[:8]
    )

    status: str = "idle"

    processed_images: int = 0

    failed_images: int = 0

    active_task: Optional[str] = None

    is_available: bool = True

    # --------------------------------------------------------------------------
    # DIRECTORIES
    # --------------------------------------------------------------------------

    input_dir: Optional[Path] = None

    output_dir: Optional[Path] = None

    temp_dir: Optional[Path] = None

    log_dir: Optional[Path] = None


# ==============================================================================
# GPU WORKER MANAGER
# ==============================================================================

class GPUWorkerManager:

    def __init__(self):

        self.worker_root = Path(

            GPU_WORKER_CONFIG[
                "worker_root_dir"
            ]
        )

        self.shared_output_dir = Path(

            GPU_WORKER_CONFIG[
                "shared_output_dir"
            ]
        )

        self.shared_output_dir.mkdir(

            parents=True,

            exist_ok=True
        )

        self.workers: Dict[int, GPUWorker] = {}

        self.task_queue = queue.Queue(

            maxsize=GPU_WORKER_CONFIG[
                "max_queue_size"
            ]
        )

        self.completed_tasks = []

        self.failed_tasks = []

        self.worker_locks = {}

        logger.info(
            "GPU Worker Manager initialized"
        )

    # ==========================================================================
    # INITIALIZE GPU WORKERS
    # ==========================================================================

    def initialize_workers(self):

        logger.info("=" * 60)
        logger.info(
            "INITIALIZING GPU WORKERS"
        )
        logger.info("=" * 60)

        try:

            gpu_count = (
                RUNTIME_STATE["gpu_count"]
            )

            available_gpus = (
                RUNTIME_STATE[
                    "available_gpus"
                ]
            )

            if gpu_count == 0:

                logger.warning(
                    "No GPU detected. "
                    "Worker initialization skipped."
                )

                return

            for gpu_info in available_gpus:

                worker = GPUWorker(

                    worker_id=gpu_info["index"],

                    gpu_id=gpu_info["index"],

                    gpu_name=gpu_info["name"],

                    vram_gb=gpu_info["vram_gb"],
                )

                self.prepare_worker_directories(
                    worker
                )

                self.workers[
                    worker.worker_id
                ] = worker

                self.worker_locks[
                    worker.worker_id
                ] = threading.Lock()

                logger.info(
                    f"Worker initialized | "
                    f"Worker={worker.worker_id} | "
                    f"GPU={worker.gpu_name} | "
                    f"VRAM={worker.vram_gb}GB"
                )

            logger.info(
                f"Total workers initialized: "
                f"{len(self.workers)}"
            )

        except Exception:

            logger.exception(
                "GPU worker initialization failed"
            )

    # ==========================================================================
    # PREPARE WORKER DIRECTORIES
    # ==========================================================================

    def prepare_worker_directories(
        self,
        worker: GPUWorker
    ):

        worker_dir = (

            self.worker_root /

            f"gpu{worker.gpu_id}"
        )

        worker.input_dir = (
            worker_dir / "inputs"
        )

        worker.output_dir = (
            worker_dir / "outputs"
        )

        worker.temp_dir = (
            worker_dir / "temp"
        )

        worker.log_dir = (
            worker_dir / "logs"
        )

        directories = [

            worker_dir,

            worker.input_dir,

            worker.output_dir,

            worker.temp_dir,

            worker.log_dir,
        ]

        for directory in directories:

            directory.mkdir(

                parents=True,

                exist_ok=True
            )

        logger.info(
            f"Worker directories ready: "
            f"gpu{worker.gpu_id}"
        )

    # ==========================================================================
    # GET AVAILABLE WORKER
    # ==========================================================================

    def get_available_worker(
        self
    ) -> Optional[GPUWorker]:

        for worker in self.workers.values():

            if (
                worker.is_available
                and
                worker.status == "idle"
            ):

                return worker

        return None

    # ==========================================================================
    # ASSIGN TASK TO WORKER
    # ==========================================================================

    def assign_task(
        self,
        worker_id: int,
        task_name: str
    ):

        worker = self.workers.get(
            worker_id
        )

        if not worker:

            logger.error(
                f"Worker not found: "
                f"{worker_id}"
            )

            return

        worker.status = "busy"

        worker.active_task = task_name

        logger.info(
            f"Task assigned | "
            f"Worker={worker_id} | "
            f"Task={task_name}"
        )

    # ==========================================================================
    # RELEASE WORKER
    # ==========================================================================

    def release_worker(
        self,
        worker_id: int,
        success: bool = True
    ):

        worker = self.workers.get(
            worker_id
        )

        if not worker:

            return

        if success:

            worker.processed_images += 1

        else:

            worker.failed_images += 1

        logger.info(
            f"Worker released | "
            f"Worker={worker_id} | "
            f"Success={success}"
        )

        worker.status = "idle"

        worker.active_task = None

    # ==========================================================================
    # GET WORKER TEMP PATHS
    # ==========================================================================

    def get_worker_paths(
        self,
        worker_id: int
    ) -> Optional[dict]:

        worker = self.workers.get(
            worker_id
        )

        if not worker:

            return None

        return {

            "input_dir": worker.input_dir,

            "output_dir": worker.output_dir,

            "temp_dir": worker.temp_dir,

            "log_dir": worker.log_dir,
        }

    # ==========================================================================
    # CLEAN WORKER DIRECTORIES
    # ==========================================================================

    def clean_worker_directories(
        self,
        worker_id: Optional[int] = None
    ):

        logger.info("=" * 60)
        logger.info(
            "CLEANING WORKER DIRECTORIES"
        )
        logger.info("=" * 60)

        target_workers = []

        if worker_id is not None:

            worker = self.workers.get(
                worker_id
            )

            if worker:

                target_workers.append(worker)

        else:

            target_workers = list(
                self.workers.values()
            )

        for worker in target_workers:

            for directory in [

                worker.input_dir,

                worker.output_dir,

                worker.temp_dir,
            ]:

                try:

                    for item in directory.iterdir():

                        if item.is_file():

                            item.unlink()

                except Exception:

                    logger.exception(
                        f"Failed worker cleanup: "
                        f"{directory}"
                    )

        logger.info(
            "Worker cleanup completed"
        )

    # ==========================================================================
    # GPU HEALTH CHECK
    # ==========================================================================

    def perform_health_check(self):

        if not GPU_WORKER_CONFIG[
            "enable_worker_health_check"
        ]:

            return

        logger.info("=" * 60)
        logger.info(
            "GPU WORKER HEALTH CHECK"
        )
        logger.info("=" * 60)

        try:

            if not torch.cuda.is_available():

                logger.warning(
                    "CUDA unavailable during "
                    "health check"
                )

                return

            for worker in self.workers.values():

                try:

                    gpu_id = worker.gpu_id

                    gpu_name = (
                        torch.cuda.get_device_name(
                            gpu_id
                        )
                    )

                    allocated_memory = round(

                        torch.cuda.memory_allocated(
                            gpu_id
                        ) / (1024 ** 3),

                        2
                    )

                    reserved_memory = round(

                        torch.cuda.memory_reserved(
                            gpu_id
                        ) / (1024 ** 3),

                        2
                    )

                    logger.info(
                        f"GPU Health | "
                        f"GPU={gpu_name} | "
                        f"Allocated={allocated_memory}GB | "
                        f"Reserved={reserved_memory}GB"
                    )

                except Exception:

                    logger.exception(
                        f"Health check failed for "
                        f"GPU {worker.gpu_id}"
                    )

        except Exception:

            logger.exception(
                "Global health check failed"
            )

    # ==========================================================================
    # QUEUE TASK
    # ==========================================================================

    def enqueue_task(
        self,
        task_data: dict
    ):

        try:

            self.task_queue.put(
                task_data,
                timeout=5
            )

            logger.info(
                f"Task queued: "
                f"{task_data.get('task_name')}"
            )

        except Exception:

            logger.exception(
                "Failed to enqueue task"
            )

    # ==========================================================================
    # DEQUEUE TASK
    # ==========================================================================

    def dequeue_task(self):

        try:

            return self.task_queue.get_nowait()

        except queue.Empty:

            return None

    # ==========================================================================
    # WORKER SUMMARY
    # ==========================================================================

    def print_worker_summary(self):

        logger.info("=" * 60)
        logger.info(
            "GPU WORKER SUMMARY"
        )
        logger.info("=" * 60)

        if not self.workers:

            logger.warning(
                "No workers initialized"
            )

            return

        for worker in self.workers.values():

            logger.info(

                f"Worker {worker.worker_id} | "

                f"GPU={worker.gpu_name} | "

                f"Status={worker.status} | "

                f"Processed={worker.processed_images} | "

                f"Failed={worker.failed_images}"
            )

    # ==========================================================================
    # GET TOTAL WORKERS
    # ==========================================================================

    def get_total_workers(self):

        return len(self.workers)

    # ==========================================================================
    # GET ACTIVE_WORKERS
    # ==========================================================================

    def get_active_workers(self):

        return [

            worker

            for worker in self.workers.values()

            if worker.status == "busy"
        ]

    # ==========================================================================
    # GET IDLE WORKERS
    # ==========================================================================

    def get_idle_workers(self):

        return [

            worker

            for worker in self.workers.values()

            if worker.status == "idle"
        ]


# ==============================================================================
# GLOBAL GPU WORKER MANAGER INSTANCE
# ==============================================================================

gpu_worker_manager = GPUWorkerManager()


# ==============================================================================
# INITIALIZE GPU WORKERS
# ==============================================================================

gpu_worker_manager.initialize_workers()

gpu_worker_manager.perform_health_check()

gpu_worker_manager.print_worker_summary()


logger.info("=" * 60)
logger.info(
    "GPU WORKER MANAGER INITIALIZED"
)
logger.info("=" * 60)

[2026-05-18 12:12:07][INFO] ============================================================
[2026-05-18 12:12:07][INFO] INITIALIZING GPU WORKER MANAGER
[2026-05-18 12:12:07][INFO] ============================================================
[2026-05-18 12:12:07][INFO] GPU Worker Manager initialized
[2026-05-18 12:12:07][INFO] ============================================================
[2026-05-18 12:12:07][INFO] INITIALIZING GPU WORKERS
[2026-05-18 12:12:07][INFO] ============================================================
[2026-05-18 12:12:07][INFO] Worker directories ready: gpu0
[2026-05-18 12:12:07][INFO] Worker initialized | Worker=0 | GPU=Tesla T4 | VRAM=14.56GB
[2026-05-18 12:12:07][INFO] Worker directories ready: gpu1
[2026-05-18 12:12:07][INFO] Worker initialized | Worker=1 | GPU=Tesla T4 | VRAM=14.56GB
[2026-05-18 12:12:07][INFO] Total workers initialized: 2
[2026-05-18 12:12:07][INFO] ============================================================
[2026-05-18 12:12:07][INFO] GPU 

# 📁 5. Data Manager & Backend Logic

In [5]:
# ==============================================================================
# ADVANCED DATA MANAGER & BACKEND LOGIC
# Multi GPU Ready + Worker Isolation + Resume Safe Architecture
# Optimized for Kaggle / Colab + Open Source Scalability
# ==============================================================================

import gc
import json
import os
import re
import shutil
import threading
import time
import zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from pathlib import Path

import cv2
import gdown
import gradio as gr
import torch
from PIL import Image


logger.info("=" * 60)
logger.info(
    "INITIALIZING ADVANCED DATA MANAGER"
)
logger.info("=" * 60)


# ==============================================================================
# DIRECTORY CONFIGURATION
# ==============================================================================

INPUT_DIR = Path("gradio_inputs")

OUTPUT_DIR = Path("gradio_outputs")

REPORT_DIR = Path("reports")

CACHE_DIR = Path("cache")

INPUT_DIR.mkdir(exist_ok=True)

OUTPUT_DIR.mkdir(exist_ok=True)

REPORT_DIR.mkdir(exist_ok=True)

CACHE_DIR.mkdir(exist_ok=True)


# ==============================================================================
# FILE CONFIGURATION
# ==============================================================================

ALLOWED_EXTENSIONS = {

    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp",
    ".tiff",
}

PROCESSED_DB = "processed_files.json"

FAILED_DB = "failed_files.json"


# ==============================================================================
# BACKEND CONFIGURATION
# ==============================================================================

BACKEND_CONFIG = {

    "enable_resume_system": True,

    "enable_failed_tracking": True,

    "enable_processing_report": True,

    "enable_worker_isolation": True,

    "enable_auto_cleanup": True,

    "enable_runtime_metrics": True,

    "enable_zip_optimization": True,

    "max_zip_size_warning_gb": 4,

    "batch_thread_pool_size": 4,
}


# ==============================================================================
# PROCESSING REPORT SYSTEM
# ==============================================================================

class ProcessingReportManager:

    @staticmethod
    def generate_report(
        report_data: dict
    ):

        if not BACKEND_CONFIG[
            "enable_processing_report"
        ]:

            return

        try:

            timestamp = datetime.now().strftime(
                "%Y%m%d_%H%M%S"
            )

            report_path = (

                REPORT_DIR /

                f"processing_report_{timestamp}.json"
            )

            with open(
                report_path,
                "w"
            ) as file:

                json.dump(

                    report_data,

                    file,

                    indent=2
                )

            logger.info(
                f"Processing report saved: "
                f"{report_path.name}"
            )

        except Exception:

            logger.exception(
                "Failed generating processing report"
            )


# ==============================================================================
# RESUME SYSTEM (THREAD-SAFE THREADING LOCK)
# ==============================================================================

import threading
import json

db_lock = threading.Lock()

def load_json_database(file_path: str) -> set:
    if not os.path.exists(file_path):
        return set()
        
    try:
        if os.path.getsize(file_path) == 0:
            return set()
            
        with open(file_path, "r") as file:
            return set(json.load(file))
            
    except json.JSONDecodeError:
        logger.warning(f"Database corrupted or empty, resetting: {file_path}")
        return set()
    except Exception:
        logger.exception(f"Failed loading database: {file_path}")
        return set()

def save_json_entry(file_path: str, value: str):
    with db_lock:  # Ensure only one thread writes at a time
        try:
            current_data = load_json_database(file_path)
            current_data.add(value)
            
            with open(file_path, "w") as file:
                json.dump(list(current_data), file, indent=2)
                
        except Exception:
            logger.exception(f"Failed saving database entry: {value}")

def load_processed_files():
    with db_lock:
        return load_json_database(PROCESSED_DB)

def save_processed_file(filename):
    save_json_entry(PROCESSED_DB, filename)

def save_failed_file(filename):
    save_json_entry(FAILED_DB, filename)


# ==============================================================================
# DIRECTORY CLEANUP
# ==============================================================================

def clear_folder(folder_path):

    folder_path = Path(folder_path)

    if not folder_path.exists():

        return

    logger.info(
        f"Clearing folder: {folder_path}"
    )

    for item in folder_path.iterdir():

        try:

            if item.is_file():

                item.unlink()

            elif item.is_dir():

                shutil.rmtree(item)

        except Exception:

            logger.exception(
                f"Failed deleting item: {item}"
            )


# ==============================================================================
# IMAGE VALIDATION SYSTEM
# ==============================================================================

def validate_image_file(
    image_path: Path
) -> bool:

    try:

        with Image.open(image_path) as image:

            image.verify()

        image = cv2.imread(
            str(image_path)
        )

        return image is not None

    except Exception:

        return False


def clean_and_verify_images(
    folder: Path
):

    accepted_stats = Counter()

    rejected_stats = Counter()

    logger.info(
        f"Scanning images: {folder}"
    )

    for path in folder.rglob("*"):

        if not path.is_file():

            continue

        extension = (

            path.suffix.lower()

            or "no_ext"
        )

        try:

            with Image.open(path) as image:

                image_format = image.format

                image.verify()

            normalized_extension = (

                f".{image_format.lower()}"

                if image_format

                else extension
            )

            if (
                normalized_extension
                not in ALLOWED_EXTENSIONS
            ):

                raise ValueError(
                    "Unsupported format"
                )

            # --------------------------------------------------------------
            # AUTO EXTENSION FIX
            # --------------------------------------------------------------

            if (
                extension
                != normalized_extension
            ):

                new_path = path.with_suffix(
                    normalized_extension
                )

                path.rename(new_path)

                logger.info(
                    f"Extension fixed: "
                    f"{path.name} → "
                    f"{new_path.name}"
                )

                path = new_path

            # --------------------------------------------------------------
            # FINAL VALIDATION
            # --------------------------------------------------------------

            if not validate_image_file(path):

                raise ValueError(
                    "Invalid image content"
                )

            accepted_stats[
                normalized_extension
            ] += 1

        except Exception:

            rejected_stats[
                extension
            ] += 1

            logger.warning(
                f"Rejected invalid image: "
                f"{path.name}"
            )

            try:

                path.unlink()

            except Exception:

                logger.exception(
                    f"Failed deleting invalid image: "
                    f"{path}"
                )

    logger.info(
        f"Validation complete | "
        f"Accepted={sum(accepted_stats.values())} | "
        f"Rejected={sum(rejected_stats.values())}"
    )

    return accepted_stats, rejected_stats


# ==============================================================================
# GALLERY SYSTEM
# ==============================================================================

def get_gallery_images(
    folder=INPUT_DIR
):

    folder = Path(folder)

    if not folder.exists():

        return []

    return sorted([

        str(path)

        for path in folder.iterdir()

        if (
            path.is_file()
            and
            path.suffix.lower()
            in ALLOWED_EXTENSIONS
        )
    ])


# ==============================================================================
# ZIP EXTRACTION SYSTEM
# ==============================================================================

def extract_zip_archive(
    zip_path: Path,
    target_dir: Path
):

    logger.info(
        f"Extracting ZIP archive: "
        f"{zip_path.name}"
    )

    shutil.unpack_archive(

        str(zip_path),

        str(target_dir)
    )


# ==============================================================================
# FILE IMPORT SYSTEM
# ==============================================================================

def import_single_file(
    source_path: Path,
    destination_dir: Path
):

    target_path = (

        destination_dir /
        source_path.name
    )

    shutil.copy(
        str(source_path),
        str(target_path)
    )

    # --------------------------------------------------------------------------
    # ZIP EXTRACTION
    # --------------------------------------------------------------------------

    if zipfile.is_zipfile(target_path):

        extract_zip_archive(
            target_path,
            destination_dir
        )

        target_path.unlink()


# ==============================================================================
# FILE UPLOAD HANDLER
# ==============================================================================

def handle_upload(files):

    if not files:

        return (
            "❌ No uploaded files detected.",
            get_gallery_images(INPUT_DIR)
        )

    logger.info(
        f"Uploading {len(files)} file(s)"
    )

    try:

        with ThreadPoolExecutor(

            max_workers=BACKEND_CONFIG[
                "batch_thread_pool_size"
            ]

        ) as executor:

            futures = []

            for file_object in files:

                futures.append(

                    executor.submit(

                        import_single_file,

                        Path(file_object.name),

                        INPUT_DIR
                    )
                )

            for future in futures:

                future.result()

        accepted, rejected = (

            clean_and_verify_images(
                INPUT_DIR
            )
        )

        return (

            f"✅ Upload completed | "
            f"Valid={sum(accepted.values())} | "
            f"Rejected={sum(rejected.values())}",

            get_gallery_images(INPUT_DIR)
        )

    except Exception:

        logger.exception(
            "Upload handler failed"
        )

        return (
            "❌ Upload failed.",
            get_gallery_images(INPUT_DIR)
        )


# ==============================================================================
# KAGGLE IMPORT SYSTEM
# ==============================================================================

def handle_kaggle_path(path_string):

    source_path = Path(path_string.strip())

    if not source_path.exists():

        return (

            f"❌ Path not found: {source_path}",

            get_gallery_images(INPUT_DIR)
        )

    logger.info(
        f"Loading Kaggle path: "
        f"{source_path}"
    )

    try:

        if source_path.is_file():

            import_single_file(
                source_path,
                INPUT_DIR
            )

        else:

            files = [

                file

                for file in source_path.iterdir()

                if file.is_file()
            ]

            with ThreadPoolExecutor(

                max_workers=BACKEND_CONFIG[
                    "batch_thread_pool_size"
                ]

            ) as executor:

                futures = []

                for file in files:

                    futures.append(

                        executor.submit(

                            import_single_file,

                            file,

                            INPUT_DIR
                        )
                    )

                for future in futures:

                    future.result()

        accepted, rejected = (

            clean_and_verify_images(
                INPUT_DIR
            )
        )

        return (

            f"✅ Kaggle import completed | "
            f"Valid={sum(accepted.values())}",

            get_gallery_images(INPUT_DIR)
        )

    except Exception:

        logger.exception(
            "Kaggle import failed"
        )

        return (

            "❌ Kaggle import failed.",

            get_gallery_images(INPUT_DIR)
        )


# ==============================================================================
# GOOGLE DRIVE IMPORT
# ==============================================================================

def handle_gdrive(url):

    url = url.strip()

    if not url:

        return (
            "❌ Empty Google Drive URL.",
            get_gallery_images(INPUT_DIR)
        )

    logger.info(
        "Starting Google Drive import"
    )

    try:

        temp_download = (
            CACHE_DIR / "gdrive_temp"
        )

        # ----------------------------------------------------------------------
        # FOLDER
        # ----------------------------------------------------------------------

        if "folders/" in url:

            folder_match = re.search(

                r"folders/([a-zA-Z0-9_-]+)",

                url
            )

            if folder_match:

                gdown.download_folder(

                    id=folder_match.group(1),

                    output=str(INPUT_DIR),

                    quiet=True
                )

        # ----------------------------------------------------------------------
        # SINGLE FILE
        # ----------------------------------------------------------------------

        else:

            file_match = re.search(

                r"(?:d/|id=)([a-zA-Z0-9_-]+)",

                url
            )

            if file_match:

                gdown.download(

                    id=file_match.group(1),

                    output=str(temp_download),

                    quiet=True
                )

            else:

                gdown.download(

                    url,

                    output=str(temp_download),

                    quiet=True,

                    fuzzy=True
                )

            if (
                temp_download.exists()
                and
                zipfile.is_zipfile(temp_download)
            ):

                extract_zip_archive(

                    temp_download,

                    INPUT_DIR
                )

                temp_download.unlink()

        accepted, rejected = (

            clean_and_verify_images(
                INPUT_DIR
            )
        )

        return (

            f"✅ Google Drive import completed | "
            f"Valid={sum(accepted.values())}",

            get_gallery_images(INPUT_DIR)
        )

    except Exception:

        logger.exception(
            "Google Drive import failed"
        )

        return (

            "❌ Google Drive import failed.",

            get_gallery_images(INPUT_DIR)
        )


# ==============================================================================
# RESET HANDLERS
# ==============================================================================

def handle_reset_input():

    clear_folder(INPUT_DIR)

    logger.info(
        "Input directory reset completed"
    )

    return (

        "🗑️ Input directory cleared.",

        []
    )


def handle_reset_output():

    clear_folder(OUTPUT_DIR)

    logger.info(
        "Output directory reset completed"
    )

    return (
        "🗑️ Output directory cleared.",
        [],
        gr.update(value=None, visible=False)
    )


# ==============================================================================
# PNG PREPROCESSING
# ==============================================================================

def preprocess_png_if_needed(
    image_path: Path
):

    if image_path.suffix.lower() != ".png":

        return image_path

    try:

        with Image.open(image_path) as image:

            if image.mode in (

                "RGBA",
                "LA",
                "P"
            ):

                rgb_image = image.convert(
                    "RGB"
                )

                rgb_image.save(image_path)

                logger.info(
                    f"PNG converted to RGB: "
                    f"{image_path.name}"
                )

    except Exception:

        logger.exception(
            f"PNG preprocessing failed: "
            f"{image_path.name}"
        )

    return image_path


logger.info("=" * 60)
logger.info(
    "ADVANCED DATA MANAGER INITIALIZED"
)
logger.info("=" * 60)

[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] INITIALIZING ADVANCED DATA MANAGER
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] ADVANCED DATA MANAGER INITIALIZED
[2026-05-18 12:12:14][INFO] ============================================================


# 🗃️ 6. Batch Prosesing Engine

In [6]:
# ==============================================================================
# ADVANCED BATCH PROCESSING ENGINE
# Multi GPU Parallel Processing + Worker Orchestration + Queue System
# Optimized for Kaggle / Colab + Open Source Scalability
# ==============================================================================

import gc
import json
import queue
import shutil
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

import gradio as gr
import torch


logger.info("=" * 60)
logger.info(
    "INITIALIZING ADVANCED BATCH ENGINE"
)
logger.info("=" * 60)


# ==============================================================================
# BATCH ENGINE CONFIGURATION
# ==============================================================================

BATCH_ENGINE_CONFIG = {

    # --------------------------------------------------------------------------
    # MULTI GPU
    # --------------------------------------------------------------------------

    "enable_multi_gpu": True,

    "enable_parallel_processing": True,

    # --------------------------------------------------------------------------
    # BATCH
    # --------------------------------------------------------------------------

    "max_parallel_workers": 8,

    "batch_queue_timeout": 10,

    "worker_retry_attempts": 3,

    # --------------------------------------------------------------------------
    # CLEANUP
    # --------------------------------------------------------------------------

    "enable_memory_cleanup": True,

    "enable_worker_cleanup": True,

    # --------------------------------------------------------------------------
    # REPORT
    # --------------------------------------------------------------------------

    "enable_runtime_metrics": True,

    "enable_processing_report": True,
}


# ==============================================================================
# TASK DATA STRUCTURE
# ==============================================================================

@dataclass
class BatchTask:

    task_id: str

    task_index: int
    
    total_tasks: int

    image_path: Path

    output_folder: Path

    model_type: str

    scale: int

    quality: str

    gpu_id: int

    worker_id: int

    status: str = "pending"

    start_time: Optional[float] = None

    end_time: Optional[float] = None

    elapsed_time: float = 0

    output_path: Optional[Path] = None

    error_message: Optional[str] = None


# ==============================================================================
# BATCH PROCESSING ENGINE
# ==============================================================================

class ParallelBatchProcessor:

    def __init__(self):
        self.task_queue = queue.Queue()
        self.completed_tasks = []
        self.failed_tasks = []
        self.processing_tasks = []
        self.runtime_metrics = {
            "start_time": None,
            "end_time": None,
            "total_runtime": 0,
            "processed_images": 0,
            "failed_images": 0,
        }
        self.queue_lock = threading.Lock()
        logger.info("Parallel Batch Processor initialized")

    # ==========================================================================
    # TASK GENERATION
    # ==========================================================================
    def generate_tasks(self, files, model_type, scale, quality):
        logger.info("=" * 60)
        logger.info("GENERATING BATCH TASKS")
        logger.info("=" * 60)

        workers = list(gpu_worker_manager.workers.values())
        if not workers:
            raise RuntimeError("No GPU workers available")

        generated_tasks = []
        worker_index = 0
        total_tasks = len(files)

        for index, image_file in enumerate(files, start=1):
            worker = workers[worker_index % len(workers)]
            
            task = BatchTask(
                task_id=f"task_{index}",
                task_index=index,
                total_tasks=total_tasks,
                image_path=image_file,
                output_folder=OUTPUT_DIR,
                model_type=model_type,
                scale=scale,
                quality=quality,
                gpu_id=worker.gpu_id,
                worker_id=worker.worker_id,
            )
            generated_tasks.append(task)
            worker_index += 1

        logger.info(f"Generated {len(generated_tasks)} batch task(s)")
        return generated_tasks

    # ==========================================================================
    # SINGLE TASK EXECUTION
    # ==========================================================================
    def process_single_task(self, task: BatchTask):
        worker = gpu_worker_manager.workers.get(task.worker_id)
        progress_str = f"[{task.task_index}/{task.total_tasks}]"

        if not worker:
            task.status = "failed"
            task.error_message = "Worker not found"
            return task

        logger.info("=" * 60)
        logger.info(f"PROCESSING TASK {progress_str}: {task.task_id}")
        logger.info("=" * 60)
        
        logger.info(
            f"Worker={task.worker_id} | "
            f"GPU={task.gpu_id} | "
            f"Image={task.image_path.name}"
        )

        task.start_time = time.time()
        gpu_worker_manager.assign_task(worker.worker_id, task.image_path.name)

        try:
            gpu_worker_manager.clean_worker_directories(worker.worker_id)
            worker_paths = gpu_worker_manager.get_worker_paths(worker.worker_id)
            isolated_input_dir = worker_paths["input_dir"]
            isolated_output_dir = worker_paths["output_dir"]
            
            isolated_input_path = isolated_input_dir / task.image_path.name
            shutil.copy(str(task.image_path), str(isolated_input_path))
            isolated_input_path = preprocess_png_if_needed(isolated_input_path)

            result = upscale_image(
                image_path=isolated_input_path,
                output_folder=task.output_folder,
                isolated_output_dir=isolated_output_dir,
                image_type="anime" if task.model_type == "Anime / Cartoon" else "general",
                scale=int(task.scale),
                quality=task.quality.lower(),
                gpu_id=task.gpu_id
            )

            task.end_time = time.time()
            task.elapsed_time = task.end_time - task.start_time

            if result and Path(result).exists():
                task.status = "completed"
                task.output_path = Path(result)
                save_processed_file(task.image_path.name)
                logger.info(
                    f"Task completed {progress_str} | "
                    f"{task.image_path.name} | "
                    f"{task.elapsed_time:.2f}s"
                )
                gpu_worker_manager.release_worker(worker.worker_id, success=True)
            else:
                task.status = "failed"
                task.error_message = "No valid output generated"
                save_failed_file(task.image_path.name)
                logger.error(f"Task failed {progress_str} | {task.image_path.name}")
                gpu_worker_manager.release_worker(worker.worker_id, success=False)

        except Exception as error:
            task.status = "failed"
            task.error_message = str(error)
            save_failed_file(task.image_path.name)
            logger.exception(f"Fatal task exception {progress_str}: {task.image_path.name}")
            gpu_worker_manager.release_worker(worker.worker_id, success=False)

        finally:
            if BATCH_ENGINE_CONFIG["enable_memory_cleanup"]:
                try:
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                except Exception:
                    logger.exception("Memory cleanup failed")

            if BATCH_ENGINE_CONFIG["enable_worker_cleanup"]:
                try:
                    gpu_worker_manager.clean_worker_directories(worker.worker_id)
                except Exception:
                    logger.exception("Worker cleanup failed")

        return task

    # ==========================================================================
    # PARALLEL EXECUTION
    # ==========================================================================

    def execute_parallel_batch(
        self,
        tasks,
        progress=None
    ):

        logger.info("=" * 60)
        logger.info(
            "STARTING PARALLEL BATCH EXECUTION"
        )
        logger.info("=" * 60)

        self.runtime_metrics[
            "start_time"
        ] = time.time()

        completed_tasks = []

        max_workers = min(

            len(gpu_worker_manager.workers),

            BATCH_ENGINE_CONFIG[
                "max_parallel_workers"
            ]
        )

        logger.info(
            f"Parallel workers: {max_workers}"
        )

        # ----------------------------------------------------------------------
        # PARALLEL EXECUTION
        # ----------------------------------------------------------------------

        with ThreadPoolExecutor(

            max_workers=max_workers

        ) as executor:

            futures = []

            for task in tasks:

                futures.append(

                    executor.submit(

                        self.process_single_task,

                        task
                    )
                )

            for index, future in enumerate(

                futures,

                start=1
            ):

                result_task = future.result()

                completed_tasks.append(
                    result_task
                )

                # --------------------------------------------------------------
                # PROGRESS UPDATE
                # --------------------------------------------------------------

                if progress:

                    progress(

                        index / len(tasks),

                        desc=(
                            f"Processing "
                            f"{index}/{len(tasks)}"
                        )
                    )

        self.runtime_metrics[
            "end_time"
        ] = time.time()

        self.runtime_metrics[
            "total_runtime"
        ] = (

            self.runtime_metrics[
                "end_time"
            ]

            -

            self.runtime_metrics[
                "start_time"
            ]
        )

        self.runtime_metrics[
            "processed_images"
        ] = len([

            task

            for task in completed_tasks

            if task.status == "completed"
        ])

        self.runtime_metrics[
            "failed_images"
        ] = len([

            task

            for task in completed_tasks

            if task.status == "failed"
        ])

        logger.info(
            "Parallel batch execution completed"
        )

        return completed_tasks

    # ==========================================================================
    # PROCESSING REPORT
    # ==========================================================================

    def generate_processing_report(
        self,
        completed_tasks
    ):

        successful_tasks = [

            task

            for task in completed_tasks

            if task.status == "completed"
        ]

        failed_tasks = [

            task

            for task in completed_tasks

            if task.status == "failed"
        ]

        average_time = 0

        if successful_tasks:

            average_time = round(

                sum(
                    task.elapsed_time

                    for task in successful_tasks
                )

                /

                len(successful_tasks),

                2
            )

        report_data = {

            "timestamp": datetime.now().isoformat(),

            "runtime_metrics": self.runtime_metrics,

            "successful_tasks": len(successful_tasks),

            "failed_tasks": len(failed_tasks),

            "average_processing_time": average_time,

            "successful_files": [

                task.image_path.name

                for task in successful_tasks
            ],

            "failed_files": [

                {

                    "filename": task.image_path.name,

                    "error": task.error_message
                }

                for task in failed_tasks
            ]
        }

        ProcessingReportManager.generate_report(

            report_data
        )

    # ==========================================================================
    # ZIP EXPORT
    # ==========================================================================

    def create_result_zip():

        logger.info(
            "Creating ZIP archive"
        )

        zip_filename = (

            f"RealESRGAN_Results_"
            f"{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        )

        zip_path = Path(
            f"{zip_filename}.zip"
        )

        if zip_path.exists():

            zip_path.unlink()

        shutil.make_archive(

            zip_filename,

            "zip",

            str(OUTPUT_DIR)
        )

        logger.info(
            f"ZIP archive created: "
            f"{zip_path.name}"
        )

        return zip_path


# ==============================================================================
# MAIN BATCH UPSCALE FUNCTION
# ==============================================================================

def run_batch_upscale(
    model_type,
    scale,
    quality,
    auto_zip_enabled,
    auto_download_enabled,
    progress=gr.Progress()
):

    logger.info("=" * 60)
    logger.info(
        "STARTING ADVANCED BATCH UPSCALE"
    )
    logger.info("=" * 60)

    try:

        # ----------------------------------------------------------------------
        # INPUT VALIDATION
        # ----------------------------------------------------------------------

        if (
            not INPUT_DIR.exists()
            or
            not list(INPUT_DIR.glob("*"))
        ):

            logger.warning(
                "No input images detected"
            )

            return (

                "⛔ No images found.",

                get_gallery_images(OUTPUT_DIR),

                gr.update(
                    visible=False
                )
            )

        # ----------------------------------------------------------------------
        # LOAD RESUME DATABASE
        # ----------------------------------------------------------------------

        processed_files = (
            load_processed_files()
        )

        files = [

            file

            for file in INPUT_DIR.iterdir()

            if (
                file.suffix.lower()
                in ALLOWED_EXTENSIONS
                and
                file.name
                not in processed_files
            )
        ]

        total_files = len(files)

        logger.info(
            f"Filtered files for processing: "
            f"{total_files}"
        )

        if total_files == 0:

            return (

                "✅ All files already processed.",

                get_gallery_images(OUTPUT_DIR),

                gr.update(
                    visible=False
                )
            )

        # ----------------------------------------------------------------------
        # INITIALIZE PROCESSOR
        # ----------------------------------------------------------------------

        processor = (
            ParallelBatchProcessor()
        )

        # ----------------------------------------------------------------------
        # TASK GENERATION
        # ----------------------------------------------------------------------

        tasks = processor.generate_tasks(

            files=files,

            model_type=model_type,

            scale=scale,

            quality=quality
        )

        # ----------------------------------------------------------------------
        # EXECUTION
        # ----------------------------------------------------------------------

        completed_tasks = (

            processor.execute_parallel_batch(

                tasks,

                progress=progress
            )
        )

        # ----------------------------------------------------------------------
        # REPORT
        # ----------------------------------------------------------------------

        processor.generate_processing_report(

            completed_tasks
        )

        # ----------------------------------------------------------------------
        # FINAL SUMMARY
        # ----------------------------------------------------------------------

        success_count = len([

            task

            for task in completed_tasks

            if task.status == "completed"
        ])

        failed_count = len([

            task

            for task in completed_tasks

            if task.status == "failed"
        ])

        runtime = round(

            processor.runtime_metrics[
                "total_runtime"
            ],

            2
        )

        message_log = (
            f"🚀 Batch Processing Completed\n\n"
            f"✅ Success : {success_count}\n"
            f"❌ Failed  : {failed_count}\n"
            f"⚡ Runtime : {runtime}s\n"
            f"🖥 GPU Workers : "
            f"{gpu_worker_manager.get_total_workers()}"
        )

        logger.info("=" * 60)
        logger.info(
            "ADVANCED BATCH COMPLETED"
        )
        logger.info("=" * 60)

        logger.info(
            f"Success={success_count} | "
            f"Failed={failed_count} | "
            f"Runtime={runtime}s"
        )

        # ----------------------------------------------------------------------
        # AUTO ZIP
        # ----------------------------------------------------------------------

        download_update = gr.update(

            visible=False,

            value=None
        )

        if auto_zip_enabled:

            try:

                zip_path = (

                    ParallelBatchProcessor.create_result_zip()
                )

                message_log += (
                    "\n📦 ZIP archive created."
                )

                if auto_download_enabled:

                    download_update = gr.update(

                        value=str(zip_path),

                        visible=True
                    )

            except Exception:

                logger.exception(
                    "ZIP creation failed"
                )

                message_log += (
                    "\n❌ ZIP creation failed."
                )

        return (

            message_log,

            get_gallery_images(OUTPUT_DIR),

            download_update
        )

    except Exception:

        logger.exception(
            "Fatal batch processing failure"
        )

        return (

            "❌ Fatal batch processing error.",

            get_gallery_images(OUTPUT_DIR),

            gr.update(
                visible=False
            )
        )


logger.info("=" * 60)
logger.info(
    "ADVANCED BATCH ENGINE INITIALIZED"
)
logger.info("=" * 60)

[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] INITIALIZING ADVANCED BATCH ENGINE
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] ADVANCED BATCH ENGINE INITIALIZED
[2026-05-18 12:12:14][INFO] ============================================================


# 🎨 7. Gradio UI / UX Interface

In [7]:
# ==============================================================================
# ADVANCED GRADIO UI / UX SYSTEM
# Professional Open Source Interface for Easy-RealESRGAN
# Multi GPU Ready + Batch Monitoring + Runtime Dashboard
# ==============================================================================

import shutil
from pathlib import Path

import gradio as gr


logger.info("=" * 60)
logger.info(
    "BUILDING ADVANCED GRADIO UI"
)
logger.info("=" * 60)


# ==============================================================================
# ADVANCED CUSTOM CSS
# ==============================================================================

CUSTOM_CSS = """
/* =========================================================
   GLOBAL UI
========================================================= */

body {
    overflow-x: hidden !important;
}

/* =========================================================
   GALLERY SYSTEM
========================================================= */

.strict-gallery .grid-container {
    grid-template-columns: repeat(4, minmax(0, 1fr)) !important;
}

.strict-gallery-out .grid-container {
    grid-template-columns: repeat(3, minmax(0, 1fr)) !important;
}

/* =========================================================
   TEXTBOX
========================================================= */

.hide-scrollbar textarea {
    overflow-y: hidden !important;
}

/* =========================================================
   FILE PREVIEW & UPLOAD COMPONENT
========================================================= */

/* Memaksa kontainer utama Gradio File agar memiliki batas tinggi */
.scrollable-file {
    max-height: 260px !important;
    overflow-y: auto !important;
    border: 1px solid var(--border-color-primary) !important;
}

/* Menargetkan blok internal list file pada Gradio */
.scrollable-file > div,
.scrollable-file .file-preview-holder,
.scrollable-file .file-preview {
    max-height: 240px !important;
    overflow-y: auto !important;
}

/* Memastikan tabel list file (Gradio 3 & 4) tidak merusak layout */
.scrollable-file table, 
.scrollable-file tbody {
    display: block !important;
    max-height: 200px !important;
    overflow-y: auto !important;
    width: 100% !important;
}

/* Merapikan scrollbar agar terlihat profesional (Opsional untuk Webkit) */
.scrollable-file::-webkit-scrollbar,
.scrollable-file *::-webkit-scrollbar {
    width: 8px;
}
.scrollable-file::-webkit-scrollbar-thumb,
.scrollable-file *::-webkit-scrollbar-thumb {
    background-color: var(--border-color-primary);
    border-radius: 4px;
}

/* =========================================================
   STATUS BOX
========================================================= */

.status-box textarea {
    font-size: 13px !important;
    line-height: 1.5 !important;
}

/* =========================================================
   DASHBOARD
========================================================= */

.dashboard-box {
    border-radius: 12px !important;
    padding: 12px !important;
}

/* =========================================================
   BUTTONS
========================================================= */

button {
    border-radius: 10px !important;
}
"""


# ==============================================================================
# DASHBOARD HELPERS
# ==============================================================================

def get_runtime_dashboard():

    try:

        gpu_count = (
            RUNTIME_STATE["gpu_count"]
        )

        worker_count = (
            gpu_worker_manager.get_total_workers()
        )

        runtime_type = (
            RUNTIME_STATE["runtime_type"]
        )

        fp16_support = (
            RUNTIME_STATE["fp16_supported"]
        )

        gpu_info = []

        for gpu in RUNTIME_STATE["available_gpus"]:

            gpu_info.append(

                f"• GPU {gpu['index']} → "
                f"{gpu['name']} "
                f"({gpu['vram_gb']} GB)"
            )

        dashboard_text = f"""
### 🖥 Runtime Information

**Runtime:** {runtime_type}

**GPU Count:** {gpu_count}

**GPU Workers:** {worker_count}

**FP16 Support:** {fp16_support}

---

### 🚀 Active GPUs

{chr(10).join(gpu_info)}
"""

        return dashboard_text

    except Exception:

        logger.exception(
            "Failed generating runtime dashboard"
        )

        return "❌ Failed loading dashboard."


# ==============================================================================
# GALLERY SELECTION
# ==============================================================================

def on_select_gallery(evt: gr.SelectData):

    try:

        images = get_gallery_images(INPUT_DIR)

        if evt.index < len(images):

            selected_path = images[evt.index]

            logger.info(
                f"Gallery selected: "
                f"{Path(selected_path).name}"
            )

            return (

                selected_path,

                f"Selected: "
                f"{Path(selected_path).name}",

                gr.update(interactive=True)
            )

    except Exception:

        logger.exception(
            "Gallery selection failed"
        )

    return (

        None,

        "",

        gr.update(interactive=False)
    )


# ==============================================================================
# DELETE SELECTED FILE
# ==============================================================================

def on_delete_single(file_path):

    try:

        if file_path and Path(file_path).exists():

            Path(file_path).unlink()

            logger.info(
                f"Deleted image: "
                f"{Path(file_path).name}"
            )

            return (

                f"🗑️ Deleted: "
                f"{Path(file_path).name}",

                get_gallery_images(INPUT_DIR),

                None,

                "",

                gr.update(interactive=False)
            )

    except Exception:

        logger.exception(
            "Delete selected image failed"
        )

    return (

        "❌ Failed deleting selected image.",

        get_gallery_images(INPUT_DIR),

        None,

        "",

        gr.update(interactive=False)
    )


# ==============================================================================
# ZIP EXPORT HANDLER
# ==============================================================================

def handle_create_zip(
    progress=gr.Progress()
):

    try:

        progress(
            0,
            desc="Preparing ZIP archive..."
        )

        output_dir = OUTPUT_DIR

        if (
            not output_dir.exists()
            or
            not list(output_dir.iterdir())
        ):

            logger.warning(
                "ZIP export aborted: "
                "output directory empty"
            )

            return (

                "❌ Output folder is empty.",

                gr.update(
                    visible=False
                )
            )

        zip_filename = (
            "EasyRealESRGAN_Results"
        )

        zip_path = Path(
            f"{zip_filename}.zip"
        )

        if zip_path.exists():

            zip_path.unlink()

        logger.info(
            "Creating ZIP archive"
        )

        shutil.make_archive(

            zip_filename,

            "zip",

            str(output_dir)
        )

        progress(
            1.0,
            desc="ZIP archive completed!"
        )

        logger.info(
            f"ZIP archive ready: "
            f"{zip_path.name}"
        )

        return (

            "✅ ZIP archive created successfully!",

            gr.update(
                value=str(zip_path),
                visible=True
            )
        )

    except Exception:

        logger.exception(
            "ZIP export failed"
        )

        return (

            "❌ ZIP export failed.",

            gr.update(
                visible=False
            )
        )


# ==============================================================================
# MAIN UI BUILDER
# ==============================================================================

def build_ui():

    logger.info(
        "Initializing advanced Gradio interface"
    )

    theme = gr.themes.Soft(

        primary_hue="blue",

        secondary_hue="slate"
    )

    with gr.Blocks(

        theme=theme,

        title="Easy-RealESRGAN",

        css=CUSTOM_CSS

    ) as demo:

        # ======================================================================
        # HEADER
        # ======================================================================

        gr.Markdown(
            """
# 🚀 Easy-RealESRGAN Image Upscaler

Professional Open Source AI Upscaling Notebook  
Optimized for Kaggle, Colab, Batch Processing & Multi GPU
"""
        )

        # ======================================================================
        # DASHBOARD
        # ======================================================================

        with gr.Accordion(
            "🖥 Runtime Dashboard",
            open=False  # Changed from True to False (Closed by default)
        ):

            runtime_dashboard = gr.Markdown(
                value=get_runtime_dashboard(),
                elem_classes="dashboard-box"
            )

        # ======================================================================
        # MAIN TABS
        # ======================================================================

        with gr.Tabs():

            # ==================================================================
            # TAB 1 — DATA MANAGER
            # ==================================================================

            with gr.Tab("📁 Data Manager"):

                with gr.Row():

                    # ----------------------------------------------------------
                    # LEFT PANEL
                    # ----------------------------------------------------------

                    with gr.Column(scale=1):

                        # ------------------------------------------------------
                        # UPLOAD
                        # ------------------------------------------------------

                        with gr.Accordion(
                            "Upload Images / ZIP",
                            open=True
                        ):

                            upload_component = gr.File(

                                file_count="multiple",

                                label="Upload Images or ZIP",

                                elem_classes="scrollable-file"
                            )

                            button_upload = gr.Button(

                                "Upload & Extract",

                                variant="primary"
                            )

                        # ------------------------------------------------------
                        # KAGGLE DATASET
                        # ------------------------------------------------------

                        with gr.Accordion(
                            "Load from Kaggle Dataset",
                            open=False
                        ):

                            kaggle_path_component = gr.Textbox(

                                placeholder="/kaggle/input/...",

                                label="Kaggle Dataset Path"
                            )

                            button_kaggle = gr.Button(

                                "Load Dataset",

                                variant="primary"
                            )

                        # ------------------------------------------------------
                        # GOOGLE DRIVE
                        # ------------------------------------------------------

                        with gr.Accordion(
                            "Import from Google Drive",
                            open=False
                        ):

                            gdrive_component = gr.Textbox(

                                placeholder=(
                                    "https://drive.google.com/..."
                                ),

                                label="Google Drive URL"
                            )

                            button_gdrive = gr.Button(

                                "Import from GDrive",

                                variant="primary"
                            )

                        # ------------------------------------------------------
                        # STATUS
                        # ------------------------------------------------------

                        status_text_input = gr.Textbox(

                            label="Input Status",

                            interactive=False,

                            lines=2,

                            max_lines=8,

                            elem_classes=[
                                "hide-scrollbar",
                                "status-box"
                            ]
                        )

                        button_reset_input = gr.Button(

                            "🗑️ Clear Input Folder",

                            variant="stop"
                        )

                    # ----------------------------------------------------------
                    # RIGHT PANEL
                    # ----------------------------------------------------------

                    with gr.Column(scale=2):

                        gr.Markdown(
                            "### 🖼 Input Gallery"
                        )

                        gallery_input = gr.Gallery(

                            show_label=False,

                            columns=4,

                            rows=2,

                            height=450,

                            object_fit="contain",

                            allow_preview=True,

                            elem_classes="strict-gallery"
                        )

                        with gr.Row():

                            selected_file_state = gr.State(
                                None
                            )

                            selected_file_text = gr.Textbox(

                                show_label=False,

                                placeholder=(
                                    "Select image to delete..."
                                ),

                                interactive=False,

                                lines=2,

                                elem_classes="hide-scrollbar",

                                scale=4
                            )

                            button_delete_single = gr.Button(

                                "🗑️ Delete",

                                variant="stop",

                                interactive=False,

                                scale=1
                            )

            # ==================================================================
            # TAB 2 — UPSCALE PLAYGROUND
            # ==================================================================

            with gr.Tab("⚙️ Batch Upscale Playground"):

                with gr.Row():

                    # ----------------------------------------------------------
                    # LEFT PANEL
                    # ----------------------------------------------------------

                    with gr.Column(scale=1):

                        gr.Markdown(
                            "### 🎛 Upscale Configuration"
                        )

                        model_type = gr.Radio(

                            [
                                "General / Photo",

                                "Anime / Cartoon"
                            ],

                            value="General / Photo",

                            label="Model Type"
                        )

                        scale = gr.Radio(

                            ["2", "4"],

                            value="4",

                            label="Scale Factor"
                        )

                        quality = gr.Dropdown(

                            choices=[
                                "ultra",
                                "high",
                                "balanced",
                                "fast"
                            ],

                            value="balanced",

                            label="Quality Preset",

                            info=(
                                "Ultra = Best Quality | "
                                "Fast = Lowest VRAM"
                            )
                        )

                        auto_zip = gr.Checkbox(

                            value=True,

                            label=(
                                "Auto Create ZIP"
                            )
                        )

                        auto_download = gr.Checkbox(

                            value=False,

                            label=(
                                "Auto Show Download"
                            )
                        )

                        # (SYSTEM INFO/Processing Info has been completely removed from here)

                        # ------------------------------------------------------
                        # RUN BUTTON
                        # ------------------------------------------------------

                        button_run_batch = gr.Button(

                            "🚀 START BATCH UPSCALE",

                            variant="primary",

                            size="lg"
                        )

                        # ------------------------------------------------------
                        # STATUS OUTPUT
                        # ------------------------------------------------------

                        status_text_output = gr.Textbox(

                            label="Processing Log",

                            interactive=False,

                            lines=6,

                            max_lines=20,

                            elem_classes=[
                                "hide-scrollbar",
                                "status-box"
                            ]
                        )

                        with gr.Row():

                            button_create_zip = gr.Button(

                                "📦 Create ZIP",

                                variant="secondary"
                            )

                            button_reset_output = gr.Button(

                                "🗑️ Clear Output",

                                variant="stop"
                            )

                    # ----------------------------------------------------------
                    # RIGHT PANEL
                    # ----------------------------------------------------------

                    with gr.Column(scale=2):

                        gr.Markdown(
                            "### ✨ Output Gallery"
                        )

                        gallery_output = gr.Gallery(

                            show_label=False,

                            columns=3,

                            rows=2,

                            height=500,

                            object_fit="contain",

                            allow_preview=True,

                            elem_classes="strict-gallery-out"
                        )

                        download_zip = gr.File(

                            label="Download ZIP",

                            interactive=False,

                            visible=False
                        )

        # ======================================================================
        # EVENT REGISTRATION
        # ======================================================================

        logger.info(
            "Registering Gradio events"
        )

        # ----------------------------------------------------------------------
        # INPUT HANDLERS
        # ----------------------------------------------------------------------

        button_upload.click(

            handle_upload,

            inputs=[upload_component],

            outputs=[
                status_text_input,
                gallery_input
            ]
        )

        button_kaggle.click(

            handle_kaggle_path,

            inputs=[kaggle_path_component],

            outputs=[
                status_text_input,
                gallery_input
            ]
        )

        button_gdrive.click(

            handle_gdrive,

            inputs=[gdrive_component],

            outputs=[
                status_text_input,
                gallery_input
            ]
        )

        button_reset_input.click(

            handle_reset_input,

            outputs=[
                status_text_input,
                gallery_input
            ]
        )

        # ----------------------------------------------------------------------
        # GALLERY EVENTS
        # ----------------------------------------------------------------------

        gallery_input.select(

            on_select_gallery,

            outputs=[
                selected_file_state,
                selected_file_text,
                button_delete_single
            ]
        )

        button_delete_single.click(

            on_delete_single,

            inputs=[selected_file_state],

            outputs=[
                status_text_input,
                gallery_input,
                selected_file_state,
                selected_file_text,
                button_delete_single
            ]
        )

        # ----------------------------------------------------------------------
        # UPSCALE EVENTS
        # ----------------------------------------------------------------------

        button_run_batch.click(

            run_batch_upscale,

            inputs=[
                model_type,
                scale,
                quality,
                auto_zip,
                auto_download
            ],

            outputs=[
                status_text_output,
                gallery_output,
                download_zip
            ]
        )

        # ----------------------------------------------------------------------
        # ZIP EVENTS
        # ----------------------------------------------------------------------

        button_create_zip.click(

            handle_create_zip,

            outputs=[
                status_text_output,
                download_zip
            ]
        )

        # ----------------------------------------------------------------------
        # OUTPUT RESET
        # ----------------------------------------------------------------------

        button_reset_output.click(

            handle_reset_output,

            outputs=[
                status_text_output,
                gallery_output,
                download_zip
            ]
        )

    logger.info(
        "Advanced Gradio UI initialized successfully"
    )

    return demo


logger.info("=" * 60)
logger.info(
    "ADVANCED UI/UX SYSTEM INITIALIZED"
)
logger.info("=" * 60)

[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] BUILDING ADVANCED GRADIO UI
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] ADVANCED UI/UX SYSTEM INITIALIZED
[2026-05-18 12:12:14][INFO] ============================================================


# 🌐 8. Cloudflare Tunnel Launcher

In [8]:
# ==============================================================================
# ADVANCED CLOUDFLARE TUNNEL & APPLICATION LAUNCHER
# Persistent Public Gradio Access for Kaggle / Colab Runtime
# Optimized for Stability + Open Source Deployment
# ==============================================================================

import os
import queue
import re
import signal
import socket
import subprocess
import threading
import time
import urllib.request
from pathlib import Path

from IPython.display import HTML, display


logger.info("=" * 60)
logger.info(
    "INITIALIZING ADVANCED CLOUDFLARE SYSTEM"
)
logger.info("=" * 60)


# ==============================================================================
# CLOUDFLARE CONFIGURATION
# ==============================================================================

CLOUDFLARED_BINARY = Path(
    "cloudflared"
)

CLOUDFLARED_DOWNLOAD_URL = (
    "https://github.com/cloudflare/cloudflared/"
    "releases/latest/download/cloudflared-linux-amd64"
)

CLOUDFLARE_CONFIG = {

    "enable_infinite_execution": True,

    "startup_timeout": 60,

    "keep_alive_interval": 30,

    "max_tunnel_wait": 60,

    "enable_keep_alive": True,

    "enable_debug_log": False,
}

# ==============================================================================
# EXECUTION MODE DETECTION
# ==============================================================================

def should_run_infinitely() -> bool:
    """
    Determine if the script should block the main thread infinitely.
    Auto-detects Kaggle 'Save & Run All' (Batch) mode to prevent hanging commits.
    """
    kaggle_run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
    
    if kaggle_run_type == "Batch":
        logger.info(
            "Kaggle Batch (Commit) mode detected. "
            "Infinite execution automatically disabled."
        )
        return False
        
    return CLOUDFLARE_CONFIG.get("enable_infinite_execution", True)


# ==============================================================================
# GLOBAL RUNTIME STATE
# ==============================================================================

LAUNCHER_STATE = {

    "gradio_port": None,

    "cloudflare_url": None,

    "cloudflared_process": None,

    "gradio_started": False,

    "tunnel_started": False,
}


# ==============================================================================
# PORT MANAGEMENT
# ==============================================================================

def get_free_port():

    try:

        with socket.socket(

            socket.AF_INET,

            socket.SOCK_STREAM

        ) as socket_instance:

            socket_instance.bind(("", 0))

            port = socket_instance.getsockname()[1]

            logger.info(
                f"Allocated free port: {port}"
            )

            return port

    except Exception:

        logger.exception(
            "Failed allocating free port"
        )

        raise


# ==============================================================================
# CLOUDFLARED INSTALLER
# ==============================================================================

def ensure_cloudflared_installed():

    try:

        if CLOUDFLARED_BINARY.exists():

            logger.info(
                "Cloudflared already installed"
            )

            return

        logger.info(
            "Downloading cloudflared binary..."
        )

        urllib.request.urlretrieve(

            CLOUDFLARED_DOWNLOAD_URL,

            str(CLOUDFLARED_BINARY)
        )

        os.chmod(
            CLOUDFLARED_BINARY,
            0o777
        )

        logger.info(
            "Cloudflared installed successfully"
        )

    except Exception:

        logger.exception(
            "Cloudflared installation failed"
        )

        raise


# ==============================================================================
# CLOUDFLARE TUNNEL STARTER
# ==============================================================================

def start_cloudflare_tunnel(
    port,
    url_queue
):

    logger.info(
        "Starting Cloudflare Tunnel..."
    )

    cmd = [

        f"./{CLOUDFLARED_BINARY.name}",

        "tunnel",

        "--url",

        f"http://127.0.0.1:{port}"
    ]

    try:

        process = subprocess.Popen(

            cmd,

            stdout=subprocess.PIPE,

            stderr=subprocess.STDOUT,

            text=True,

            bufsize=1
        )

        LAUNCHER_STATE[
            "cloudflared_process"
        ] = process

        url_found = False

        for line in iter(

            process.stdout.readline,

            ""
        ):

            clean_line = line.strip()

            # --------------------------------------------------------------
            # DEBUG LOG
            # --------------------------------------------------------------

            if CLOUDFLARE_CONFIG[
                "enable_debug_log"
            ]:

                logger.info(
                    f"[Cloudflared] {clean_line}"
                )

            # --------------------------------------------------------------
            # URL DETECTION
            # --------------------------------------------------------------

            if not url_found:

                matches = re.findall(

                    r"(https://[^\s]+trycloudflare\.com)",

                    clean_line
                )

                if matches:

                    public_url = matches[0].strip()

                    LAUNCHER_STATE[
                        "cloudflare_url"
                    ] = public_url

                    LAUNCHER_STATE[
                        "tunnel_started"
                    ] = True

                    url_queue.put(public_url)

                    url_found = True

                    logger.info(
                        f"Cloudflare URL detected: "
                        f"{public_url}"
                    )

            # --------------------------------------------------------------
            # IMPORTANT LOGS
            # --------------------------------------------------------------

            important_keywords = [

                "Registered tunnel",

                "Connection",

                "ERR",

                "error",
            ]

            if any(

                keyword.lower()
                in
                clean_line.lower()

                for keyword in important_keywords
            ):

                logger.info(
                    f"[Cloudflared] {clean_line}"
                )

        logger.warning(
            "Cloudflare process terminated"
        )

    except Exception:

        logger.exception(
            "Cloudflare tunnel startup failed"
        )

        url_queue.put(None)


# ==============================================================================
# KEEP ALIVE SYSTEM
# ==============================================================================

def keep_runtime_alive():

    logger.info(
        "Keep-alive system initialized"
    )

    try:

        while True:

            time.sleep(

                CLOUDFLARE_CONFIG[
                    "keep_alive_interval"
                ]
            )

            logger.info(
                "Runtime keep-alive heartbeat"
            )

    except Exception:

        logger.exception(
            "Keep-alive system failed"
        )


# ==============================================================================
# HTML URL DISPLAY
# ==============================================================================

def display_public_url(
    public_url
):

    display(

        HTML(f"""

        <div style="
            padding: 20px;
            border-radius: 14px;
            background: linear-gradient(135deg,#1e3c72,#2a5298);
            color: white;
            font-family: Arial;
            margin: 15px 0;
            box-shadow: 0 4px 12px rgba(0,0,0,0.25);
        ">

            <h2 style="margin-top:0;">
                🚀 Easy-RealESRGAN is Online
            </h2>

            <p style="font-size:15px;">
                Public application URL:
            </p>

            <a
                href="{public_url}"
                target="_blank"
                style="
                    color:#FFD700;
                    font-size:18px;
                    font-weight:bold;
                    text-decoration:none;
                    word-break:break-all;
                "
            >
                {public_url}
            </a>

            <hr style="
                border:none;
                border-top:1px solid rgba(255,255,255,0.25);
                margin:14px 0;
            ">

            <p style="
                font-size:13px;
                opacity:0.85;
                margin-bottom:0;
            ">
                Keep this notebook running to maintain public access.
            </p>

        </div>

        """)
    )


# ==============================================================================
# MAIN APPLICATION LAUNCHER
# ==============================================================================

def launch_application():

    try:

        logger.info("=" * 60)
        logger.info(
            "LAUNCHING EASY-REALESRGAN"
        )
        logger.info("=" * 60)

        # ----------------------------------------------------------------------
        # INSTALL CLOUDFLARED
        # ----------------------------------------------------------------------

        ensure_cloudflared_installed()

        # ----------------------------------------------------------------------
        # ALLOCATE PORT
        # ----------------------------------------------------------------------

        free_port = get_free_port()

        LAUNCHER_STATE[
            "gradio_port"
        ] = free_port

        logger.info(
            f"Selected port: {free_port}"
        )

        # ----------------------------------------------------------------------
        # BUILD UI
        # ----------------------------------------------------------------------

        logger.info(
            "Building Gradio interface..."
        )

        ui = build_ui()

        # ----------------------------------------------------------------------
        # START CLOUDFLARE FIRST
        # ----------------------------------------------------------------------

        url_queue = queue.Queue()

        tunnel_thread = threading.Thread(

            target=start_cloudflare_tunnel,

            args=(free_port, url_queue),

            daemon=True
        )

        tunnel_thread.start()

        logger.info(
            "Waiting for Cloudflare URL..."
        )

        try:

            public_url = url_queue.get(

                timeout=CLOUDFLARE_CONFIG[
                    "max_tunnel_wait"
                ]
            )

        except queue.Empty:

            logger.error(
                "Cloudflare URL timeout"
            )

            return

        if not public_url:

            logger.error(
                "Failed retrieving public URL"
            )

            return

        # ----------------------------------------------------------------------
        # DISPLAY PUBLIC URL
        # ----------------------------------------------------------------------

        display_public_url(
            public_url
        )

        # ----------------------------------------------------------------------
        # START KEEP ALIVE
        # ----------------------------------------------------------------------

        if CLOUDFLARE_CONFIG[
            "enable_keep_alive"
        ]:

            keep_alive_thread = threading.Thread(

                target=keep_runtime_alive,

                daemon=True
            )

            keep_alive_thread.start()

        # ----------------------------------------------------------------------
        # START GRADIO
        # ----------------------------------------------------------------------

        logger.info(
            "Launching Gradio application..."
        )

        LAUNCHER_STATE[
            "gradio_started"
        ] = True

        ui.queue().launch(

            server_name="0.0.0.0",

            server_port=free_port,

            share=False,

            inline=False,

            quiet=True,

            prevent_thread_lock=True
        )

        logger.info("=" * 60)
        logger.info(
            "APPLICATION LAUNCHED SUCCESSFULLY"
        )
        logger.info("=" * 60)

        logger.info(
            f"Public URL: {public_url}"
        )

        # ----------------------------------------------------------------------
        # KEEP NOTEBOOK RUNNING
        # ----------------------------------------------------------------------

        if should_run_infinitely():
            logger.info("Entering infinite execution mode. Press Stop to terminate.")
            while True:
                time.sleep(60)
        else:
            logger.info("Infinite loop bypassed. Script execution completed gracefully.")

    except KeyboardInterrupt:
        logger.warning("Application interrupted manually")

    except Exception:
        logger.exception("Fatal launcher failure")

    finally:

        # ----------------------------------------------------------------------
        # CLEANUP
        # ----------------------------------------------------------------------

        try:

            process = LAUNCHER_STATE.get(
                "cloudflared_process"
            )

            if process:

                logger.info(
                    "Terminating cloudflared process"
                )

                process.terminate()

        except Exception:

            logger.exception(
                "Failed terminating cloudflared"
            )


# ==============================================================================
# EXECUTION
# ==============================================================================

if __name__ == "__main__":

    launch_application()

[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] INITIALIZING ADVANCED CLOUDFLARE SYSTEM
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] LAUNCHING EASY-REALESRGAN
[2026-05-18 12:12:14][INFO] ============================================================
[2026-05-18 12:12:14][INFO] Downloading cloudflared binary...
[2026-05-18 12:12:15][INFO] Cloudflared installed successfully
[2026-05-18 12:12:15][INFO] Allocated free port: 39865
[2026-05-18 12:12:15][INFO] Selected port: 39865
[2026-05-18 12:12:15][INFO] Building Gradio interface...
[2026-05-18 12:12:15][INFO] Initializing advanced Gradio interface
[2026-05-18 12:12:15][INFO] Registering Gradio events
[2026-05-18 12:12:15][INFO] Advanced Gradio UI initialized successfully
[2026-05-18 12:12:15][INFO] Starting Cloudf

[2026-05-18 12:12:19][INFO] Keep-alive system initialized
[2026-05-18 12:12:19][INFO] [Cloudflared] 2026-05-18T12:12:19Z INF Settings: map[ha-connections:1 protocol:quic url:http://127.0.0.1:39865]
[2026-05-18 12:12:19][INFO] Launching Gradio application...
[2026-05-18 12:12:19][INFO] [Cloudflared] 2026-05-18T12:12:19Z INF Tunnel connection curve preferences: [X25519MLKEM768 CurveID(65074) CurveP256] connIndex=0 event=0 ip=198.41.192.37
[2026-05-18 12:12:19][INFO] ============================================================
[2026-05-18 12:12:19][INFO] APPLICATION LAUNCHED SUCCESSFULLY
[2026-05-18 12:12:19][INFO] ============================================================
[2026-05-18 12:12:19][INFO] Public URL: https://investing-mods-parent-positions.trycloudflare.com
[2026-05-18 12:12:19][INFO] Kaggle Batch (Commit) mode detected. Infinite execution automatically disabled.
[2026-05-18 12:12:19][INFO] Infinite loop bypassed. Script execution completed gracefully.
[2026-05-18 12:12:19][

---

# ☕ Support the Project

If this project has been helpful to you, please consider supporting its development.  
Your support helps maintain and improve the project, fund future research, and continue building better open-source AI tools for the community.

Every contribution truly means a lot and helps push this project further. 🚀

| Platform | Link |
| :--- | :--- |
| **Trakteer** | https://trakteer.id/pyforge |
| **Saweria** | https://saweria.co/pyforge |
| **SociaBuzz** | https://sociabuzz.com/pyforge |
| **PayPal** | https://www.paypal.com/paypalme/Masyura |

---

# ⚖️ License

This project is licensed under the **MIT License**.

You are free to use, modify, and distribute this project, provided that the original copyright and license notice are included.

Copyright (c) 2026 **MasyuraC7**

---